In [ ]:
!nvidia-smi

Sat Oct 17 12:47:47 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   41C    P8              9W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [ ]:
import jax

print("GPU:", jax.devices())
print("Backend:", jax.default_backend())

GPU: [CudaDevice(id=0)]
Backend: gpu


In [ ]:
%cd /content
!git clone https://github.com/apirrone/Open_Duck_Playground.git
%cd /content/Open_Duck_Playground

/content
Cloning into 'Open_Duck_Playground'...
remote: Enumerating objects: 4549, done.
remote: Counting objects: 100% (1416/1416), done.
remote: Compressing objects: 100% (190/190), done.
remote: Total 4549 (delta 1348), reused 1226 (delta 1226), pack-reused 3133 (from 1)
Receiving objects: 100% (4549/4549), 76.08 MiB | 19.96 MiB/s, done.
Resolving deltas: 100% (3233/3233), done.
/content/Open_Duck_Playground


In [ ]:
!pip -q install uv
!uv sync

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 17.3/17.3 MB 119.2 MB/s eta 0:00:00
Using CPython 3.13.15 interpreter at: /usr/bin/python3
Creating virtual environment at: .venv
Resolved 130 packages in 702ms
Prepared 120 packages in 37.00s
Installed 120 packages in 645ms
 + absl-py==2.5.1
 + aiofiles==25.1.0
 + asttokens==3.0.2
 + astunparse==1.6.3
 + blinker==1.9.0
 + brax==0.14.2
 + certifi==2026.7.22
 + charset-normalizer==3.5.2
 + click==8.5.0
 + contourpy==1.4.0
 + cycler==0.12.1
 + etils==1.14.0
 + executing==2.3.0
 + flask==3.1.3
 + flask-cors==6.0.5
 + flatbuffers==25.12.19
 + flax==0.12.10
 + fonttools==4.67.0
 + framesviewer==1.1.0
 + fsspec==2026.9.0
 + gast==0.7.0
 + glfw==2.11.0
 + google-pasta==0.2.0
 + grpcio==1.84.0
 + h5py==3.14.0
 + humanize==4.16.0
 + idna==3.20
 + ipython==9.17.1
 + ipython-pygments-lexers==1.1.1
 + itsdangerous==2.2.0
 + jax==0.11.2
 + jax-cuda12-pjrt==0.11.2
 + jax-cuda12-plugin==0.11.2
 + jaxlib==0.11.2
 + jaxopt==0.8.5
 + jaxtyping==0.3.11
 + jedi=

In [ ]:
!uv run python -c "import jax; print(jax.devices()); print(jax.default_backend())"

[CudaDevice(id=0)]
gpu


In [ ]:
!uv run playground/open_duck_mini_v2/runner.py \
    --env standing \
    --num_timesteps 100000 \
    --output_dir /content/checkpoints/test

Traceback (most recent call last):
  File "/content/Open_Duck_Playground/playground/open_duck_mini_v2/runner.py", line 7, in <module>
    from playground.open_duck_mini_v2 import joystick, standing
  File "/content/Open_Duck_Playground/playground/open_duck_mini_v2/joystick.py", line 27, in <module>
    from mujoco_playground._src.collision import geoms_colliding
ModuleNotFoundError: No module named 'mujoco_playground._src.collision'


In [ ]:
%cd /content/Open_Duck_Playground

!uv run python -c "import mujoco, mujoco_playground; print('MuJoCo:', mujoco.__version__); print('Playground:', mujoco_playground.__file__)"

/content/Open_Duck_Playground
MuJoCo: 3.16.0
Playground: /content/Open_Duck_Playground/.venv/lib/python3.13/site-packages/mujoco_playground/__init__.py


In [ ]:
from pathlib import Path

root = Path("/content/Open_Duck_Playground")
common = root / "playground/common"

# 接触判定の互換関数
helper = common / "collision_compat.py"

helper.write_text("""
import jax.numpy as jnp

def geoms_colliding(data, geom1, geom2):
    contact = data.contact
    pair = (
        ((contact.geom[:, 0] == geom1) &
         (contact.geom[:, 1] == geom2))
        |
        ((contact.geom[:, 0] == geom2) &
         (contact.geom[:, 1] == geom1))
    )
    return jnp.any(pair & (contact.dist < 0))
""")

# 両方の環境のimportを修正
for name in ["standing.py", "joystick.py"]:
    path = root / "playground/open_duck_mini_v2" / name
    source = path.read_text()

    source = source.replace(
        "from mujoco_playground._src.collision import geoms_colliding",
        "from playground.common.collision_compat import geoms_colliding"
    )

    path.write_text(source)
    print("Updated:", name)

Updated: standing.py
Updated: joystick.py


In [ ]:
!uv run python -c "from playground.open_duck_mini_v2 import standing, joystick; print('Import OK')"

Import OK


In [ ]:
!uv run playground/open_duck_mini_v2/runner.py \
    --env standing \
    --num_timesteps 100000 \
    --output_dir /content/checkpoints/test

xml: /content/Open_Duck_Playground/playground/open_duck_mini_v2/xmls/scene_flat_terrain.xml
actuators: ['left_hip_yaw', 'left_hip_roll', 'left_hip_pitch', 'left_knee', 'left_ankle', 'neck_pitch', 'head_pitch', 'head_yaw', 'head_roll', 'right_hip_yaw', 'right_hip_roll', 'right_hip_pitch', 'right_knee', 'right_ankle']
joints: ['floating_base', 'left_hip_yaw', 'left_hip_roll', 'left_hip_pitch', 'left_knee', 'left_ankle', 'neck_pitch', 'head_pitch', 'head_yaw', 'head_roll', 'right_hip_yaw', 'right_hip_roll', 'right_hip_pitch', 'right_knee', 'right_ankle']
backlash joints: []
actuator joints ids: [1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14]
actuator joints dict: {'left_hip_yaw': 1, 'left_hip_roll': 2, 'left_hip_pitch': 3, 'left_knee': 4, 'left_ankle': 5, 'neck_pitch': 6, 'head_pitch': 7, 'head_yaw': 8, 'head_roll': 9, 'right_hip_yaw': 10, 'right_hip_roll': 11, 'right_hip_pitch': 12, 'right_knee': 13, 'right_ankle': 14}
floating qpos addr: 0 qvel addr: 0
xml: /content/Open_Duck_Playground

In [ ]:
%cd /content/Open_Duck_Playground

!git restore playground/open_duck_mini_v2/standing.py
!git restore playground/open_duck_mini_v2/joystick.py

/content/Open_Duck_Playground


In [ ]:
from pathlib import Path

p = Path("/content/Open_Duck_Playground/pyproject.toml")

s = p.read_text()
s = s.replace(
    '"playground>=0.0.3"',
    '"playground==0.0.5"'
)

p.write_text(s)

print("Updated pyproject.toml")

Updated pyproject.toml


In [ ]:
!uv sync

Resolved 129 packages in 332ms
Prepared 2 packages in 734ms
Uninstalled 3 packages in 15ms
Installed 2 packages in 4ms
 ~ open-duck-playground==0.1.0 (from file:///content/Open_Duck_Playground)
 - playground==0.2.0
 + playground==0.0.5
 - warp-lang==1.18.0


In [ ]:
!uv run python -c "
import jax
import importlib.metadata
from mujoco_playground._src import mjx_env
from mujoco_playground._src.collision import geoms_colliding

print('Playground:', importlib.metadata.version('playground'))
print('JAX backend:', jax.default_backend())
print('mjx_env.init:', hasattr(mjx_env, 'init'))
print('Collision: OK')
"

SyntaxError: unterminated string literal (detected at line 11) (2052280965.py, line 11)

In [ ]:
%cd /content/Open_Duck_Playground

!uv run python -c "import jax; import importlib.metadata; from mujoco_playground._src import mjx_env; from mujoco_playground._src.collision import geoms_colliding; print('Playground:', importlib.metadata.version('playground')); print('JAX backend:', jax.default_backend()); print('mjx_env.init:', hasattr(mjx_env, 'init')); print('Collision: OK')"

In [ ]:
%cd /content/Open_Duck_Playground

!uv run playground/open_duck_mini_v2/runner.py \
    --env standing \
    --num_timesteps 100000 \
    --output_dir /content/checkpoints/test

In [ ]:
from pathlib import Path

output = Path("/content/checkpoints/test")

if output.exists():
    for p in output.rglob("*"):
        if p.is_file():
            print(p.relative_to(output), p.stat().st_size, "bytes")
else:
    print("出力ディレクトリがまだ存在しません")

In [ ]:
%cd /content/Open_Duck_Playground

!uv add "jax[cuda12]==0.9.2" "jaxlib==0.9.2"

In [ ]:
!uv run python -c "import jax, brax; print('JAX:', jax.__version__); print('Brax:', brax.__version__); print('GPU:', jax.devices()); print('Backend:', jax.default_backend()); print('Legacy API:', hasattr(jax, 'device_put_replicated'))"

In [ ]:
!uv run playground/open_duck_mini_v2/runner.py \
    --env standing \
    --num_timesteps 100000 \
    --output_dir /content/checkpoints/test

In [ ]:
from pathlib import Path

output = Path("/content/checkpoints/test")

if output.exists():
    for p in output.rglob("*"):
        if p.is_file():
            print(p.relative_to(output), p.stat().st_size, "bytes")
else:
    print("出力ディレクトリがまだ存在しません")

In [ ]:
from pathlib import Path

root = Path("/content/checkpoints/test")

models = sorted(
    root.rglob("*.onnx"),
    key=lambda p: int(p.stem.split("_")[-1])
)

print("保存モデル数:", len(models))

for p in models:
    print(
        f"step={p.stem.split('_')[-1]:>10}",
        p
    )

In [ ]:
import os
import sys
import subprocess

venv_python = (
    "/content/Open_Duck_Playground/.venv/bin/python"
)

site_packages = subprocess.check_output(
    [
        venv_python,
        "-c",
        "import site; print(site.getsitepackages()[0])"
    ],
    text=True
).strip()

sys.path.insert(0, site_packages)

os.environ["MUJOCO_GL"] = "egl"

import mujoco
import onnxruntime as ort

print("MuJoCo:", mujoco.__version__)
print("ONNX Runtime:", ort.__version__)

In [ ]:
%%writefile /content/Open_Duck_Playground/check_onnx.py
import onnxruntime as ort

model = (
    "/content/checkpoints/test/"
    "2026_10_09_045412_2293760.onnx"
)

session = ort.InferenceSession(
    model,
    providers=["CPUExecutionProvider"]
)

print("Inputs:")
for x in session.get_inputs():
    print(x.name, x.shape, x.type)

print("Outputs:")
for x in session.get_outputs():
    print(x.name, x.shape, x.type)

In [ ]:
%cd /content/Open_Duck_Playground
!uv run python check_onnx.py

In [ ]:
%%writefile /content/Open_Duck_Playground/record_standing.py
import os
os.environ["MUJOCO_GL"] = "egl"

import argparse
from pathlib import Path

import imageio.v2 as imageio
import mujoco
import numpy as np

from playground.open_duck_mini_v2.mujoco_infer import (
    MjInfer,
    USE_MOTOR_SPEED_LIMITS,
)


def record(model_path, output_path, seconds=10):
    robot = MjInfer(
        model_path="playground/open_duck_mini_v2/xmls/scene_flat_terrain.xml",
        reference_data="playground/open_duck_mini_v2/data/polynomial_coefficients.pkl",
        onnx_model_path=model_path,
        standing=True,
    )

    # Colab向けオフスクリーン描画
    renderer = mujoco.Renderer(robot.model, height=480, width=640)

    output_path = Path(output_path)
    output_path.parent.mkdir(parents=True, exist_ok=True)

    sim_fps = 1.0 / robot.sim_dt
    video_fps = 30
    total_steps = int(seconds * sim_fps)
    frame_interval = round(sim_fps / video_fps)

    with imageio.get_writer(
        str(output_path),
        fps=video_fps,
        codec="libx264",
        quality=8,
    ) as writer:
        for step in range(total_steps):
            mujoco.mj_step(robot.model, robot.data)

            if (step + 1) % robot.decimation == 0:
                obs = robot.get_obs(robot.data, robot.commands)
                action = np.asarray(
                    robot.policy.infer(obs), dtype=np.float64
                )

                robot.last_last_last_action = (
                    robot.last_last_action.copy()
                )
                robot.last_last_action = robot.last_action.copy()
                robot.last_action = action.copy()

                targets = (
                    robot.default_actuator
                    + action * robot.action_scale
                )

                if USE_MOTOR_SPEED_LIMITS:
                    max_delta = (
                        robot.max_motor_velocity
                        * robot.sim_dt
                        * robot.decimation
                    )
                    targets = np.clip(
                        targets,
                        robot.prev_motor_targets - max_delta,
                        robot.prev_motor_targets + max_delta,
                    )
                    robot.prev_motor_targets = targets.copy()

                robot.motor_targets = targets
                robot.data.ctrl[:] = targets

            if step % frame_interval == 0:
                renderer.update_scene(robot.data)
                writer.append_data(renderer.render())

    renderer.close()
    print("Saved:", output_path)


if __name__ == "__main__":
    parser = argparse.ArgumentParser()
    parser.add_argument("--model", required=True)
    parser.add_argument("--output", default="/content/standing_preview.mp4")
    parser.add_argument("--seconds", type=float, default=10)
    args = parser.parse_args()

    record(args.model, args.output, args.seconds)

In [ ]:
%cd /content/Open_Duck_Playground
!uv add imageio imageio-ffmpeg

In [ ]:
!MUJOCO_GL=egl uv run python record_standing.py \
    --model /content/checkpoints/test/2026_10_09_045412_2293760.onnx \
    --output /content/standing_preview.mp4 \
    --seconds 10

In [ ]:
from pathlib import Path

path = Path(
    "/content/Open_Duck_Playground/"
    "playground/open_duck_mini_v2/mujoco_infer.py"
)

source = path.read_text()

# 念のためバックアップ
backup = path.with_suffix(".py.bak")
if not backup.exists():
    backup.write_text(source)

# get_obs 内の観測項目だけを修正
start = source.index("    def get_obs(")
end = source.index("    def key_callback(", start)

before = source[start:end]
after = before.replace(
    "                self.motor_targets,\n", ""
).replace(
    "                self.imitation_phase,\n", ""
)

assert before != after, "修正対象が見つかりません"
assert "                self.motor_targets,\n" in before
assert "                self.imitation_phase,\n" in before

source = source[:start] + after + source[end:]
path.write_text(source)

print("観測次元の修正完了")

In [ ]:
%cd /content/Open_Duck_Playground

!MUJOCO_GL=egl uv run python record_standing.py \
    --model /content/checkpoints/test/2026_10_09_045412_2293760.onnx \
    --output /content/standing_preview.mp4 \
    --seconds 10

In [ ]:
from IPython.display import Video, display

display(Video(
    "/content/standing_preview.mp4",
    embed=True,
    width=640
))

In [ ]:
from google.colab import drive
from pathlib import Path
import shutil

drive.mount("/content/drive")

src = Path("/content/checkpoints/test")
dst = Path("/content/drive/MyDrive/OpenDuckRL/standing_test")

dst.parent.mkdir(parents=True, exist_ok=True)
shutil.copytree(src, dst, dirs_exist_ok=True)

shutil.copy2(
    "/content/standing_preview.mp4",
    "/content/drive/MyDrive/OpenDuckRL/standing_preview.mp4"
)

print("バックアップ完了")

In [ ]:
%cd /content/Open_Duck_Playground

!uv run playground/open_duck_mini_v2/runner.py \
    --env joystick \
    --task flat_terrain \
    --num_timesteps 1000000 \
    --output_dir /content/checkpoints/joystick_test

In [ ]:
from pathlib import Path

root = Path("/content/checkpoints/joystick_test")

models = sorted(
    root.glob("*.onnx"),
    key=lambda p: int(p.stem.split("_")[-1])
)

print("保存モデル数:", len(models))

for p in models:
    print(p.stem.split("_")[-1], p.name)

In [ ]:
from pathlib import Path

path = Path(
    "/content/Open_Duck_Playground/"
    "playground/open_duck_mini_v2/mujoco_infer.py"
)

backup = path.with_suffix(".py.bak")

if backup.exists():
    path.write_text(backup.read_text())
    print("公式の推論コードを復元しました")
else:
    print("バックアップがありません。元ファイルを確認してください")

In [ ]:
%cd /content/Open_Duck_Playground

!uv run python -c "import numpy as np; from playground.open_duck_mini_v2.mujoco_infer import MjInfer; r=MjInfer('playground/open_duck_mini_v2/xmls/scene_flat_terrain.xml','playground/open_duck_mini_v2/data/polynomial_coefficients.pkl','/content/checkpoints/joystick_test/2026_10_09_052202_2293760.onnx',False); print('Observation:',r.get_obs(r.data,r.commands).shape)"

In [ ]:
%%writefile /content/Open_Duck_Playground/record_joystick.py
import os
os.environ["MUJOCO_GL"] = "egl"

import argparse
from pathlib import Path

import imageio.v2 as imageio
import mujoco
import numpy as np

from playground.open_duck_mini_v2.mujoco_infer import (
    MjInfer,
    USE_MOTOR_SPEED_LIMITS,
)


def record(model_path, output_path, seconds=10):
    robot = MjInfer(
        model_path="playground/open_duck_mini_v2/xmls/scene_flat_terrain.xml",
        reference_data="playground/open_duck_mini_v2/data/polynomial_coefficients.pkl",
        onnx_model_path=str(model_path),
        standing=False,
    )
    start_xy = robot.data.qpos[:2].copy()
    renderer = None

    try:
        obs = robot.get_obs(robot.data, robot.commands)
        expected = robot.policy.ort_session.get_inputs()[0].shape[1]

        if len(obs) != expected:
            raise ValueError(
                f"Observation mismatch: {len(obs)} != {expected}"
            )

        renderer = mujoco.Renderer(
            robot.model, height=480, width=640
        )

        output_path = Path(output_path)
        output_path.parent.mkdir(parents=True, exist_ok=True)

        fps = 30
        total_steps = int(seconds / robot.model.opt.timestep)
        next_frame_time = 0.0

        with imageio.get_writer(
            str(output_path),
            fps=fps,
            codec="libx264",
            quality=8,
        ) as writer:

            for step in range(total_steps):
                t = step * robot.model.opt.timestep

                # 一定速度で前進するテスト
                command = (0.15, 0.0, 0.0)
                robot.commands[:3] = command

                mujoco.mj_step(robot.model, robot.data)

                if (step + 1) % robot.decimation == 0:
                    # 公式推論処理と同様に歩行位相を更新
                    robot.imitation_i += robot.phase_frequency_factor
                    robot.imitation_i %= robot.PRM.nb_steps_in_period

                    phase = (
                        robot.imitation_i
                        / robot.PRM.nb_steps_in_period
                        * 2 * np.pi
                    )
                    robot.imitation_phase = np.array([
                        np.cos(phase),
                        np.sin(phase),
                    ])

                    obs = robot.get_obs(
                        robot.data, robot.commands
                    )
                    action = np.asarray(
                        robot.policy.infer(obs),
                        dtype=np.float64,
                    )

                    robot.last_last_last_action = (
                        robot.last_last_action.copy()
                    )
                    robot.last_last_action = (
                        robot.last_action.copy()
                    )
                    robot.last_action = action.copy()

                    targets = (
                        robot.default_actuator
                        + action * robot.action_scale
                    )

                    if USE_MOTOR_SPEED_LIMITS:
                        max_delta = (
                            robot.max_motor_velocity
                            * robot.sim_dt
                            * robot.decimation
                        )
                        targets = np.clip(
                            targets,
                            robot.prev_motor_targets - max_delta,
                            robot.prev_motor_targets + max_delta,
                        )
                        robot.prev_motor_targets = targets.copy()

                    robot.motor_targets = targets.copy()
                    robot.data.ctrl[:] = targets

                if robot.data.time + 1e-9 >= next_frame_time:
                    renderer.update_scene(robot.data)
                    writer.append_data(renderer.render())
                    next_frame_time += 1 / fps

        # シミュレーション終了後の位置を取得
        end_xy = robot.data.qpos[:2].copy()

        delta = end_xy - start_xy
        distance = np.linalg.norm(delta)

        print("Start XY:", start_xy)
        print("End XY:", end_xy)
        print(f"Delta X: {delta[0]:.4f} m")
        print(f"Delta Y: {delta[1]:.4f} m")
        print(f"Displacement: {distance:.4f} m")

        print("Saved:", output_path)

    finally:
        if renderer is not None:
            renderer.close()


if __name__ == "__main__":
    parser = argparse.ArgumentParser()
    parser.add_argument("--model", required=True)
    parser.add_argument("--output", required=True)
    parser.add_argument("--seconds", type=float, default=10)
    args = parser.parse_args()

    record(args.model, args.output, args.seconds)

In [ ]:
from pathlib import Path
import subprocess

root = Path("/content/checkpoints/joystick_test")
out = Path("/content/joystick_videos")
out.mkdir(exist_ok=True)

targets = {
    "00_untrained": 0,
    "01_early": 327680,
    "02_middle": 1146880,
    "03_best_reward": 2129920,
    "04_final": 2293760,
}

models = {
    int(p.stem.split("_")[-1]): p
    for p in root.glob("*.onnx")
}

for name, step in targets.items():
    if step not in models:
        raise FileNotFoundError(f"Checkpoint missing: {step}")

    print(f"\nRendering {name}: step={step}", flush=True)

    subprocess.run(
        [
            "/content/Open_Duck_Playground/.venv/bin/python",
            "record_joystick.py",
            "--model", str(models[step]),
            "--output", str(out / f"{name}.mp4"),
            "--seconds", "10",
        ],
        cwd="/content/Open_Duck_Playground",
        env={**__import__("os").environ, "MUJOCO_GL": "egl"},
        check=True,
    )

print("All videos completed")

In [ ]:
from pathlib import Path
import shutil

src = Path("/content/joystick_videos")
dst = Path("/content/drive/MyDrive/OpenDuckRL/joystick_videos")

shutil.copytree(src, dst, dirs_exist_ok=True)

model_src = Path("/content/checkpoints/joystick_test")
model_dst = Path("/content/drive/MyDrive/OpenDuckRL/joystick_test")

shutil.copytree(model_src, model_dst, dirs_exist_ok=True)

print("動画と学習モデルのバックアップ完了")

In [ ]:
%cd /content/Open_Duck_Playground

!MUJOCO_GL=egl uv run python record_joystick.py \
    --model /content/checkpoints/joystick_test/2026_10_09_052152_2129920.onnx \
    --output /content/joystick_videos/03_best_reward.mp4 \
    --seconds 10

In [ ]:
%cd /content/Open_Duck_Playground

!grep -n -E \
"def run|def get_obs|decimation|commands|imitation_i|motor_targets|mj_step" \
playground/open_duck_mini_v2/mujoco_infer.py

In [ ]:
!grep -n -E \
"command|velocity|tracking|reward|alive" \
playground/open_duck_mini_v2/joystick.py

In [ ]:
%cd /content/Open_Duck_Playground

!MUJOCO_GL=egl uv run python record_joystick.py \
    --model /content/checkpoints/joystick_test/2026_10_09_052152_2129920.onnx \
    --output /content/joystick_videos/forward_test.mp4 \
    --seconds 10

In [ ]:
from IPython.display import Video, display

display(Video(
    "/content/joystick_videos/forward_test.mp4",
    embed=True,
    width=640
))

In [ ]:
%%writefile /content/Open_Duck_Playground/check_policy.py
import numpy as np

from playground.open_duck_mini_v2.mujoco_infer import MjInfer

robot = MjInfer(
    model_path="playground/open_duck_mini_v2/xmls/scene_flat_terrain.xml",
    reference_data="playground/open_duck_mini_v2/data/polynomial_coefficients.pkl",
    onnx_model_path=(
        "/content/checkpoints/joystick_test/"
        "2026_10_09_052152_2129920.onnx"
    ),
    standing=False,
)

commands = {
    "stop": (0.0, 0.0, 0.0),
    "forward": (0.15, 0.0, 0.0),
    "backward": (-0.15, 0.0, 0.0),
}

actions = {}

for name, cmd in commands.items():
    command = [*cmd, 0.0, 0.0, 0.0, 0.0]
    obs = robot.get_obs(robot.data, command)

    action = np.asarray(
        robot.policy.infer(obs)
    ).reshape(-1)

    actions[name] = action

    print(f"\n{name}")
    print("Observation shape:", obs.shape)
    print("Action:", np.round(action, 4))
    print("Action norm:", np.linalg.norm(action))

print("\n--- Action differences ---")
print(
    "forward vs stop:",
    np.linalg.norm(actions["forward"] - actions["stop"])
)
print(
    "backward vs stop:",
    np.linalg.norm(actions["backward"] - actions["stop"])
)
print(
    "forward vs backward:",
    np.linalg.norm(actions["forward"] - actions["backward"])
)

In [ ]:
%cd /content/Open_Duck_Playground
!uv run python check_policy.py

In [ ]:
%cd /content/Open_Duck_Playground

!sed -n '65,110p' playground/open_duck_mini_v2/mujoco_infer.py
!sed -n '155,235p' playground/open_duck_mini_v2/mujoco_infer.py

In [ ]:
!sed -n '330,425p' playground/open_duck_mini_v2/joystick.py
!sed -n '530,620p' playground/open_duck_mini_v2/joystick.py

In [ ]:
%cd /content/Open_Duck_Playground

!sed -n '1,110p' playground/open_duck_mini_v2/mujoco_infer.py
!sed -n '485,540p' playground/open_duck_mini_v2/joystick.py
!sed -n '45,105p' playground/open_duck_mini_v2/joystick.py

In [ ]:
%cd /content/Open_Duck_Playground

!MUJOCO_GL=egl uv run python record_joystick.py \
    --model /content/checkpoints/joystick_test/2026_10_09_052152_2129920.onnx \
    --output /content/joystick_videos/forward_no_accel_offset.mp4 \
    --seconds 10

In [ ]:
%cd /content/Open_Duck_Playground

!uv run playground/open_duck_mini_v2/runner.py --help

In [ ]:
!grep -n -E \
"restore|resume|checkpoint|load|save|num_timesteps|num_evals" \
playground/open_duck_mini_v2/runner.py

In [ ]:
%cd /content/Open_Duck_Playground

!sed -n '1,220p' playground/open_duck_mini_v2/runner.py

In [ ]:
from pathlib import Path

root = Path("/content/checkpoints/joystick_test")

checkpoints = sorted(
    [p for p in root.iterdir() if p.is_dir()],
    key=lambda p: int(p.name.split("_")[-1])
)

print("チェックポイント数:", len(checkpoints))

for p in checkpoints[-3:]:
    print(f"\n{p.name}")
    for child in list(p.iterdir())[:10]:
        print(" ", child.name)

In [ ]:
%cd /content/Open_Duck_Playground

!grep -n -C 8 "restore_checkpoint_path" \
  playground/open_duck_mini_v2/runner.py

In [ ]:
!grep -n -E \
  "ppo.train|train_fn|num_timesteps|checkpoint|restore|load" \
  playground/open_duck_mini_v2/runner.py

In [ ]:
%cd /content/Open_Duck_Playground

!sed -n '55,180p' playground/open_duck_mini_v2/runner.py

In [ ]:
!grep -Rn -E \
  "restore_checkpoint_path|restore_checkpoint|restore_fn|restore_params|load_checkpoint" \
  playground/open_duck_mini_v2 \
  playground/common

In [ ]:
%cd /content/Open_Duck_Playground

!sed -n '35,135p' playground/common/runner.py

In [ ]:
%cd /content/Open_Duck_Playground

!uv run python - <<'PY'
from brax.training.agents.ppo import checkpoint

path = (
    "/content/checkpoints/joystick_test/"
    "2026_10_09_052202_2293760"
)

params = checkpoint.load(path)

print("Restore OK")
print("パラメータ要素数:", len(params))
print("観測正規化パラメータ:", type(params[0]))
print("Policyパラメータ:", type(params[1]))
print("Valueパラメータ:", type(params[2]))
PY

In [ ]:
%cd /content/Open_Duck_Playground

!uv run playground/open_duck_mini_v2/runner.py \
    --env joystick \
    --task flat_terrain \
    --num_timesteps 5000000 \
    --restore_checkpoint_path /content/checkpoints/joystick_test/2026_10_09_052202_2293760 \
    --output_dir /content/checkpoints/joystick_resume

In [ ]:
from pathlib import Path

roots = {
    "before": Path("/content/checkpoints/joystick_test"),
    "middle": Path("/content/checkpoints/joystick_resume"),
    "best": Path("/content/checkpoints/joystick_resume"),
    "final": Path("/content/checkpoints/joystick_resume"),
}

steps = {
    "before": 2293760,
    "middle": 3440640,
    "best": 6389760,
    "final": 6881280,
}

selected = {}

for name, root in roots.items():
    matches = list(root.glob(f"*_{steps[name]}.onnx"))
    if len(matches) != 1:
        raise FileNotFoundError(
            f"{name}: expected 1 model, found {len(matches)}"
        )
    selected[name] = matches[0]
    print(name, "->", matches[0])

In [ ]:
import subprocess
import os
from pathlib import Path

output_dir = Path("/content/joystick_videos/resume_compare")
output_dir.mkdir(parents=True, exist_ok=True)

for name, model in selected.items():
    print(f"\n=== {name} ===", flush=True)

    subprocess.run(
        [
            "/content/Open_Duck_Playground/.venv/bin/python",
            "record_joystick.py",
            "--model", str(model),
            "--output", str(output_dir / f"{name}.mp4"),
            "--seconds", "10",
        ],
        cwd="/content/Open_Duck_Playground",
        env={**os.environ, "MUJOCO_GL": "egl"},
        check=True,
    )

In [ ]:
from pathlib import Path

output_dir = Path("/content/joystick_videos/resume_compare")

for name in ["before", "middle", "best", "final"]:
    video = output_dir / f"{name}.mp4"

    if video.exists():
        print(f"{name}: OK ({video.stat().st_size / 1024**2:.2f} MB)")
    else:
        print(f"{name}: NOT FOUND")

In [ ]:
from IPython.display import Video, display

for name in ["before", "middle", "best", "final"]:
    print(f"\n=== {name} ===")
    display(Video(
        f"/content/joystick_videos/resume_compare/{name}.mp4",
        embed=True,
        width=640
    ))

In [ ]:
%cd /content/Open_Duck_Playground

!sed -n '430,485p' playground/open_duck_mini_v2/joystick.py
!sed -n '620,675p' playground/open_duck_mini_v2/joystick.py

In [ ]:
!sed -n '1,38p' playground/open_duck_mini_v2/runner.py
!sed -n '90,145p' playground/common/runner.py

In [ ]:
%cd /content/Open_Duck_Playground

!sed -n '105,165p' playground/open_duck_mini_v2/joystick.py
!sed -n '235,325p' playground/open_duck_mini_v2/joystick.py

In [ ]:
!uv run python -c "from brax.training.agents.ppo import networks; print('PPO networks import OK')"

In [ ]:
%%writefile /content/Open_Duck_Playground/eval_joystick_mjx.py
import argparse

import jax
import jax.numpy as jnp
import numpy as np

from brax.training.agents.ppo import checkpoint
from brax.training.agents.ppo import networks as ppo_networks
from brax.training.acme import running_statistics

from playground.open_duck_mini_v2 import joystick
from playground.common import locomotion_params


def main():
    parser = argparse.ArgumentParser()
    parser.add_argument("--checkpoint", required=True)
    parser.add_argument("--steps", type=int, default=500)
    parser.add_argument("--vx", type=float, default=0.15)
    args = parser.parse_args()

    env = joystick.Joystick(task="flat_terrain")

    params = checkpoint.load(args.checkpoint)

    config = locomotion_params.brax_ppo_config(
        "BerkeleyHumanoidJoystickFlatTerrain"
    )

    network_config = config.get("network_factory", {})

    networks = ppo_networks.make_ppo_networks(
        observation_size=env.observation_size,
        action_size=env.action_size,
        preprocess_observations_fn=running_statistics.normalize,
        **network_config,
    )

    make_policy = ppo_networks.make_inference_fn(networks)
    policy = make_policy(params, deterministic=True)

    reset = jax.jit(env.reset)
    step = jax.jit(env.step)

    state = reset(jax.random.PRNGKey(0))
    command = jnp.array(
        [args.vx, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0]
    )

    state.info["command"] = command

    contact = jnp.array([
        joystick.geoms_colliding(
            state.data,
            geom_id,
            env._floor_geom_id,
        )
        for geom_id in env._feet_geom_id
    ])

    state = state.replace(
        obs=env._get_obs(state.data, state.info, contact)
    )

    start_xy = np.asarray(state.data.qpos[:2])
    current_key = jax.random.PRNGKey(123)

    print("Initial XY:", start_xy)

    for i in range(args.steps):
        current_key, subkey = jax.random.split(current_key)

        action, _ = policy(state.obs, subkey)
        state = step(state, action)

        if i % 50 == 0:
            xy = np.asarray(state.data.qpos[:2])
            print(
                f"step={i:4d}",
                f"x={xy[0]:.4f}",
                f"y={xy[1]:.4f}",
                f"reward={float(state.reward):.4f}",
                f"done={float(state.done):.0f}",
            )

        if bool(state.done):
            print("Terminated at step:", i)
            break

    end_xy = np.asarray(state.data.qpos[:2])
    delta = end_xy - start_xy

    print("\n=== RESULT ===")
    print("Delta X:", float(delta[0]))
    print("Delta Y:", float(delta[1]))
    print("Displacement:", float(np.linalg.norm(delta)))
    print("Simulation time:", (i + 1) * env.dt)


if __name__ == "__main__":
    main()

In [ ]:
%cd /content/Open_Duck_Playground

!uv run python eval_joystick_mjx.py \
    --checkpoint /content/checkpoints/joystick_resume/2026_10_09_060929_6881280 \
    --steps 500 \
    --vx 0.15

In [ ]:
%cd /content/Open_Duck_Playground

!grep -n "locomotion_params" playground/common/runner.py

In [ ]:
from pathlib import Path

path = Path("/content/Open_Duck_Playground/eval_joystick_mjx.py")

code = path.read_text()
code = code.replace(
    "from playground.common import locomotion_params",
    "from mujoco_playground.config import locomotion_params",
)

path.write_text(code)
print("import修正完了")

In [ ]:
%cd /content/Open_Duck_Playground

!uv run python eval_joystick_mjx.py \
    --checkpoint /content/checkpoints/joystick_resume/2026_10_09_060929_6881280 \
    --steps 500 \
    --vx 0.15

In [ ]:
%cd /content/Open_Duck_Playground

!grep -n "USE_IMITATION_REWARD\|def reward_imitation\|def reward_alive\|lin_vel_x\|tracking_sigma" \
    playground/open_duck_mini_v2/joystick.py

In [ ]:
!grep -n -A 90 "^def reward_imitation" \
    playground/open_duck_mini_v2/joystick.py


In [ ]:
%cd /content/Open_Duck_Playground

!grep -R -n \
  "def reward_imitation\|def reward_tracking_lin_vel\|def reward_alive" \
  playground/open_duck_mini_v2 playground/common

In [ ]:
!sed -n '1,105p' playground/open_duck_mini_v2/joystick.py

In [ ]:
%cd /content/Open_Duck_Playground

!sed -n '1,220p' playground/open_duck_mini_v2/custom_rewards.py

In [ ]:
!sed -n '1,52p' playground/common/rewards.py
!sed -n '120,135p' playground/common/rewards.py

In [ ]:
%cd /content/Open_Duck_Playground

!sed -n '1,180p' \
    playground/common/poly_reference_motion.py

In [ ]:
!sed -n '315,380p' \
    playground/open_duck_mini_v2/joystick.py

In [ ]:
%cd /content/Open_Duck_Playground

!mkdir -p /content/reference_policies

!curl -fL \
  https://raw.githubusercontent.com/apirrone/Open_Duck_Mini/v2/BEST_WALK_ONNX.onnx \
  -o /content/reference_policies/BEST_WALK_ONNX.onnx

!curl -fL \
  https://raw.githubusercontent.com/apirrone/Open_Duck_Mini/v2/BEST_WALK_ONNX_2.onnx \
  -o /content/reference_policies/BEST_WALK_ONNX_2.onnx

In [ ]:
%cd /content/Open_Duck_Playground

In [ ]:
%%writefile /content/Open_Duck_Playground/compare_onnx.py
from pathlib import Path
import onnxruntime as ort

reference_dir = Path("/content/reference_policies")
trained_dir = Path("/content/checkpoints/joystick_resume")

models = [
    reference_dir / "BEST_WALK_ONNX.onnx",
    reference_dir / "BEST_WALK_ONNX_2.onnx",
]

models += sorted(trained_dir.rglob("*.onnx"))[-2:]

for path in models:
    print(f"\n{'=' * 65}")
    print("MODEL:", path)
    print("SIZE:", round(path.stat().st_size / 1024, 1), "KB")

    session = ort.InferenceSession(
        str(path),
        providers=["CPUExecutionProvider"],
    )

    print("INPUTS:")
    for inp in session.get_inputs():
        print(" ", inp.name, inp.shape, inp.type)

    print("OUTPUTS:")
    for out in session.get_outputs():
        print(" ", out.name, out.shape, out.type)

In [ ]:
!uv run python compare_onnx.py

In [ ]:
%cd /content/Open_Duck_Playground

!MUJOCO_GL=egl uv run python record_joystick.py \
    --model /content/reference_policies/BEST_WALK_ONNX.onnx \
    --output /content/reference_policies/BEST_WALK_ONNX.mp4 \
    --seconds 10

In [ ]:
!MUJOCO_GL=egl uv run python record_joystick.py \
    --model /content/reference_policies/BEST_WALK_ONNX_2.onnx \
    --output /content/reference_policies/BEST_WALK_ONNX_2.mp4 \
    --seconds 10

In [ ]:
from IPython.display import Video, display

for name in ["BEST_WALK_ONNX", "BEST_WALK_ONNX_2"]:
    print(f"\n=== {name} ===")

    display(Video(
        f"/content/reference_policies/{name}.mp4",
        embed=True,
        width=640,
    ))

In [ ]:
%cd /content/Open_Duck_Playground

!MUJOCO_GL=egl uv run python record_joystick.py \
    --model /content/reference_policies/BEST_WALK_ONNX.onnx \
    --output /content/reference_policies/walk_test_1.mp4 \
    --seconds 10

!MUJOCO_GL=egl uv run python record_joystick.py \
    --model /content/reference_policies/BEST_WALK_ONNX_2.onnx \
    --output /content/reference_policies/walk_test_2.mp4 \
    --seconds 10

In [ ]:
from google.colab import drive
drive.mount("/content/drive")

In [ ]:
from pathlib import Path
from datetime import datetime, timezone
import subprocess
import shutil

root = Path("/content/Open_Duck_Playground")
exp = Path("/content/drive/MyDrive/OpenDuckRL/experiments/ppo_150m_v1")
exp.mkdir(parents=True, exist_ok=True)

files = [
    "playground/open_duck_mini_v2/joystick.py",
    "playground/open_duck_mini_v2/custom_rewards.py",
    "playground/open_duck_mini_v2/runner.py",
    "playground/common/rewards.py",
]

for name in files:
    dest = exp / "source" / name
    dest.parent.mkdir(parents=True, exist_ok=True)
    shutil.copy2(root / name, dest)

metadata = (
    f"created_utc={datetime.now(timezone.utc).isoformat()}\n"
    "experiment=ppo_150m_v1\n"
    "initialization=random\n"
    "restore_checkpoint_path=NONE\n"
    "requested_timesteps=150000000\n"
    "env=joystick\n"
    "task=flat_terrain\n"
)

(exp / "experiment.txt").write_text(metadata)
print("Saved:", exp)

In [ ]:
from pathlib import Path
import shutil

src = Path("/content/checkpoints")
dst = Path("/content/drive/MyDrive/OpenDuckRL/previous_checkpoints")

for name in ["joystick_test", "joystick_resume"]:
    source = src / name
    target = dst / name

    if source.exists():
        shutil.copytree(source, target, dirs_exist_ok=True)
        print(f"Backup completed: {name}")
    else:
        print(f"Not found: {name}")

In [ ]:
%cd /content/Open_Duck_Playground

In [ ]:
import subprocess
from pathlib import Path

backup_script = Path("/content/backup_joystick.py")

backup_script.write_text("""
import shutil
import time
from pathlib import Path

src = Path("/content/checkpoints/joystick_150m_v1")
dst = Path("/content/drive/MyDrive/OpenDuckRL/experiments/ppo_150m_v1/checkpoints")

while True:
    if src.exists():
        for item in src.iterdir():
            target = dst / item.name
            if not target.exists() and (
                item.is_file() or (
                    item.is_dir() and (item / "commit_success.txt").exists()
                )
            ):
                dst.mkdir(parents=True, exist_ok=True)
                if item.is_dir():
                    shutil.copytree(item, target)
                else:
                    shutil.copy2(item, target)
                print(f"Backed up: {item.name}", flush=True)

    time.sleep(120)
""")

log = open("/content/backup_joystick.log", "a")

backup_process = subprocess.Popen(
    ["python", str(backup_script)],
    stdout=log,
    stderr=subprocess.STDOUT,
    start_new_session=True,
)

print("Backup process PID:", backup_process.pid)

In [ ]:
%cd /content/Open_Duck_Playground

!uv run playground/open_duck_mini_v2/runner.py \
    --env joystick \
    --task flat_terrain \
    --num_timesteps 150000000 \
    --output_dir /content/checkpoints/joystick_150m_v1

In [ ]:
from pathlib import Path

local = Path("/content/checkpoints/joystick_150m_v1")
drive = Path("/content/drive/MyDrive/OpenDuckRL/experiments/ppo_150m_v1/checkpoints")

print("Local:", len(list(local.iterdir())) if local.exists() else 0)
print("Drive:", len(list(drive.iterdir())) if drive.exists() else 0)

In [ ]:
%cd /content/Open_Duck_Playground

In [ ]:
from pathlib import Path
import subprocess
import os

root = Path("/content/checkpoints/joystick_150m_v1")
output_dir = Path("/content/joystick_videos/ppo_150m")
output_dir.mkdir(parents=True, exist_ok=True)

models = {
    "step_32m": 32440320,
    "step_75m_best": 75694080,
    "step_140m": 140574720,
    "step_151m_final": 151388160,
}

for name, step in models.items():
    matches = list(root.glob(f"*_{step}.onnx"))

    if len(matches) != 1:
        print(f"Model not found: {name}, matches={len(matches)}")
        continue

    model = matches[0]
    video = output_dir / f"{name}.mp4"

    print(f"\n=== {name} ===", flush=True)

    result = subprocess.run(
        [
            "/content/Open_Duck_Playground/.venv/bin/python",
            "record_joystick.py",
            "--model", str(model),
            "--output", str(video),
            "--seconds", "10",
        ],
        cwd="/content/Open_Duck_Playground",
        env={**os.environ, "MUJOCO_GL": "egl"},
        capture_output=True,
        text=True,
    )

    print(result.stdout)

    if result.returncode != 0:
        print("ERROR:", result.stderr[-3000:])

In [ ]:
from pathlib import Path
import re

root = Path("/content/Open_Duck_Playground")
src = root / "record_joystick.py"
dst = root / "record_joystick_follow.py"

code = src.read_text()

target = "renderer.update_scene("
count = code.count(target)

if count == 0:
    raise RuntimeError(
        "renderer.update_scene() が見つかりません。"
    )

camera_func = '''
def _update_follow_scene(renderer, data, *args, **kwargs):
    """Follow the robot while rendering."""
    import mujoco

    camera = mujoco.MjvCamera()
    mujoco.mjv_defaultCamera(camera)

    camera.type = mujoco.mjtCamera.mjCAMERA_FREE
    camera.lookat[:] = data.qpos[:3]

    camera.distance = 2.0
    camera.azimuth = 135.0
    camera.elevation = -15.0

    kwargs["camera"] = camera
    renderer.update_scene(data, *args, **kwargs)


'''

# 最初のトップレベル関数またはクラスの前に挿入
match = re.search(r"(?m)^(?:def|class)\s+", code)

if match:
    insert_pos = match.start()
else:
    # 関数・クラスがないスクリプトにも対応
    # ただし通常のimport文が終わった位置を特定する
    lines = code.splitlines(keepends=True)
    insert_pos = 0

    for line in lines:
        if line.startswith(("import ", "from ")):
            insert_pos += len(line)
        elif insert_pos > 0:
            break

code = code[:insert_pos] + "\n" + camera_func + code[insert_pos:]

# 元の描画処理のみ変更（挿入した関数は除外）
before, after = code.split(camera_func, 1)

after = after.replace(
    target,
    "_update_follow_scene(renderer, "
)

code = before + camera_func + after

dst.write_text(code)

print("Created:", dst)
print("Replaced rendering calls:", count)

In [ ]:
%cd /content/Open_Duck_Playground

!MUJOCO_GL=egl uv run python record_joystick_follow.py \
    --model /content/checkpoints/joystick_150m_v1/2026_10_09_075517_140574720.onnx \
    --output /content/joystick_videos/ppo_150m/step_140m_follow.mp4 \
    --seconds 10

In [ ]:
from pathlib import Path

root = Path("/content/Open_Duck_Playground")

src = root / "record_joystick_follow.py"
dst = root / "record_soccer.py"

if not src.exists():
    raise FileNotFoundError(f"元のスクリプトがありません: {src}")

code = src.read_text()

old = "scene_flat_terrain.xml"
new = "scene_soccer.xml"

if old not in code:
    raise RuntimeError(
        "scene_flat_terrain.xml が見つかりません。"
        "録画スクリプトのXML指定を確認してください。"
    )

code = code.replace(old, new)
dst.write_text(code)

print("Created:", dst)
print("Exists:", dst.exists())

In [ ]:
from pathlib import Path
import xml.etree.ElementTree as ET

root = Path("/content/Open_Duck_Playground")
xml_dir = root / "playground/open_duck_mini_v2/xmls"

src = xml_dir / "scene_flat_terrain.xml"
dst = xml_dir / "scene_soccer.xml"

tree = ET.parse(src)
model = tree.getroot()

worldbody = model.find("worldbody")
if worldbody is None:
    raise RuntimeError("worldbody が見つかりません")

# --- Soccer ball ---
ball = ET.SubElement(
    worldbody, "body",
    name="soccer_ball",
    pos="0.8 0 0.06"
)

ET.SubElement(
    ball, "freejoint",
    name="soccer_ball_joint"
)

ET.SubElement(
    ball, "geom",
    name="soccer_ball_geom",
    type="sphere",
    size="0.06",
    mass="0.045",
    friction="0.7 0.005 0.0001",
    solref="0.02 1",
    rgba="0.95 0.95 0.95 1"
)

# --- Goal frame ---
goal = ET.SubElement(
    worldbody, "body",
    name="soccer_goal",
    pos="2.5 0 0"
)

def add_goal_box(name, pos, size, rgba):
    return ET.SubElement(
        goal, "geom",
        name=name,
        type="box",
        pos=pos,
        size=size,
        rgba=rgba,
        contype="1",
        conaffinity="1",
    )

# Goal width ~0.9m, height ~0.65m
add_goal_box(
    "goal_left_post",
    "0 -0.47 0.325",
    "0.018 0.018 0.325",
    "1 1 1 1"
)

add_goal_box(
    "goal_right_post",
    "0 0.47 0.325",
    "0.018 0.018 0.325",
    "1 1 1 1"
)

add_goal_box(
    "goal_crossbar",
    "0 0 0.65",
    "0.018 0.488 0.018",
    "1 1 1 1"
)

# Net visualization: non-colliding transparent surfaces
ET.SubElement(
    goal, "geom",
    name="goal_back_net",
    type="box",
    pos="0.23 0 0.325",
    size="0.005 0.47 0.325",
    rgba="0.75 0.85 1 0.23",
    contype="0",
    conaffinity="0",
)

for name, y in [("left", -0.47), ("right", 0.47)]:
    ET.SubElement(
        goal, "geom",
        name=f"goal_{name}_side_net",
        type="box",
        pos=f"0.115 {y} 0.325",
        size="0.115 0.005 0.325",
        rgba="0.75 0.85 1 0.18",
        contype="0",
        conaffinity="0",
    )

tree.write(
    dst,
    encoding="unicode",
    xml_declaration=True
)

print("Created:", dst)

In [ ]:
%cd /content/Open_Duck_Playground

In [ ]:
%cd /content/Open_Duck_Playground

import mujoco

path = "playground/open_duck_mini_v2/xmls/scene_soccer.xml"

model = mujoco.MjModel.from_xml_path(path)
data = mujoco.MjData(model)

mujoco.mj_forward(model, data)

print("MuJoCo scene loaded successfully")
print("Number of bodies:", model.nbody)
print("Number of geoms:", model.ngeom)
print("Number of actuators:", model.nu)

ball_id = mujoco.mj_name2id(
    model,
    mujoco.mjtObj.mjOBJ_BODY,
    "soccer_ball"
)

print("Ball position:", data.xpos[ball_id])

In [ ]:
%cd /content/Open_Duck_Playground

!MUJOCO_GL=egl uv run python record_soccer.py \
    --model /content/checkpoints/joystick_150m_v1/2026_10_09_075517_140574720.onnx \
    --output /content/joystick_videos/soccer_scene_test.mp4 \
    --seconds 10

In [ ]:
from IPython.display import Video, display

display(Video(
    "/content/joystick_videos/soccer_scene_test.mp4",
    embed=True,
    width=800
))

In [ ]:
from pathlib import Path
import xml.etree.ElementTree as ET

xml_path = Path(
    "/content/Open_Duck_Playground/"
    "playground/open_duck_mini_v2/xmls/scene_soccer.xml"
)

tree = ET.parse(xml_path)
root = tree.getroot()

ball = root.find(".//body[@name='soccer_ball']")

if ball is None:
    raise RuntimeError("soccer_ball が見つかりません")

# ボールをロボットに近づける
ball.set("pos", "0.4 0 0.06")

tree.write(
    xml_path,
    encoding="unicode",
    xml_declaration=True
)

print("New ball position:", ball.get("pos"))

In [ ]:
%cd /content/Open_Duck_Playground

!MUJOCO_GL=egl uv run python record_soccer.py \
    --model /content/checkpoints/joystick_150m_v1/2026_10_09_075517_140574720.onnx \
    --output /content/joystick_videos/soccer_scene_test_v2.mp4 \
    --seconds 10

In [ ]:
from IPython.display import Video, display

display(Video(
    "/content/joystick_videos/soccer_scene_test_v2.mp4",
    embed=True,
    width=800
))

In [ ]:
from pathlib import Path
from datetime import datetime
import shutil
import json
import hashlib

from google.colab import drive
drive.mount("/content/drive")

root = Path("/content/Open_Duck_Playground")
backup = Path("/content/drive/MyDrive/OpenDuckRL/soccer_140m")

for name in ("model", "source", "videos", "evaluation"):
    (backup / name).mkdir(parents=True, exist_ok=True)

step = 140574720
checkpoint_root = Path("/content/checkpoints/joystick_150m_v1")

# 対象のチェックポイントを特定
matches = [
    p for p in checkpoint_root.iterdir()
    if p.is_dir() and p.name.endswith(f"_{step}")
]

if len(matches) != 1:
    raise RuntimeError(
        f"140Mチェックポイントが特定できません: {matches}"
    )

checkpoint = matches[0]
onnx = checkpoint_root / f"{checkpoint.name}.onnx"

if not onnx.is_file():
    raise FileNotFoundError(f"ONNXが見つかりません: {onnx}")

files = {
    "source/scene_soccer.xml":
        root / "playground/open_duck_mini_v2/xmls/scene_soccer.xml",
    "source/scene_flat_terrain.xml":
        root / "playground/open_duck_mini_v2/xmls/scene_flat_terrain.xml",
    "source/record_joystick.py":
        root / "record_joystick.py",
    "source/record_joystick_follow.py":
        root / "record_joystick_follow.py",
    "source/record_soccer.py":
        root / "record_soccer.py",
    "model/walk_140m.onnx":
        onnx,
}

video_dir = Path("/content/joystick_videos")
for video in video_dir.rglob("*.mp4"):
    if "soccer" in video.name or "140m" in video.name:
        relative = video.relative_to(video_dir)
        files[f"videos/{relative}"] = video

# 先に必要ファイルの存在を確認
for dest, src in files.items():
    if not src.is_file():
        raise FileNotFoundError(f"{dest}: {src}")

# ファイルを保存
for dest, src in files.items():
    target = backup / dest
    target.parent.mkdir(parents=True, exist_ok=True)
    shutil.copy2(src, target)
    print("Saved:", dest)

# Orbax チェックポイントを保存
checkpoint_dst = backup / "model" / checkpoint.name

if not checkpoint_dst.exists():
    shutil.copytree(checkpoint, checkpoint_dst)
    print("Saved checkpoint:", checkpoint_dst)
else:
    print("Checkpoint already exists:", checkpoint_dst)

def sha256(path):
    h = hashlib.sha256()
    with open(path, "rb") as f:
        for chunk in iter(lambda: f.read(1024 * 1024), b""):
            h.update(chunk)
    return h.hexdigest()

manifest = {
    "created_at": datetime.now().isoformat(),
    "project": "Open Duck Mini V2 Soccer",
    "training": "PPO trained from random initialization",
    "checkpoint_step": step,
    "checkpoint_directory": checkpoint.name,
    "files": {
        dest: {
            "size_bytes": (backup / dest).stat().st_size,
            "sha256": sha256(backup / dest),
        }
        for dest in files
    },
}

(backup / "manifest.json").write_text(
    json.dumps(manifest, indent=2, ensure_ascii=False)
)

print("\nBackup completed:", backup)

In [ ]:
%%writefile /content/Open_Duck_Playground/eval_soccer.py
import argparse
import csv
import json
import os
import runpy
import sys
from pathlib import Path

os.environ.setdefault("MUJOCO_GL", "egl")

import mujoco
import numpy as np


def main():
    parser = argparse.ArgumentParser()
    parser.add_argument("--model", required=True)
    parser.add_argument("--output", required=True)
    parser.add_argument("--seconds", type=float, default=10.0)
    parser.add_argument(
        "--evaluation-dir",
        default="/content/joystick_videos/soccer_evaluation",
    )
    args = parser.parse_args()

    output = Path(args.output)
    output.parent.mkdir(parents=True, exist_ok=True)

    result_dir = Path(args.evaluation_dir)
    result_dir.mkdir(parents=True, exist_ok=True)

    # 既存のゴールXMLに合わせた値
    GOAL_X = 2.5
    GOAL_HALF_WIDTH = 0.47
    GOAL_HEIGHT = 0.65
    POST_HALF_THICKNESS = 0.018
    CROSSBAR_HALF_THICKNESS = 0.018
    BALL_RADIUS = 0.06

    # ボール全体が通れる有効開口
    max_abs_y = (
        GOAL_HALF_WIDTH
        - POST_HALF_THICKNESS
        - BALL_RADIUS
    )
    max_z = GOAL_HEIGHT - CROSSBAR_HALF_THICKNESS - BALL_RADIUS

    original_step = mujoco.mj_step
    trajectory = []
    goal_events = []
    tracked_model = None
    tracked_data = None
    ball_qpos_adr = None

    def get_ball_position(data):
        return np.asarray(
            data.qpos[ball_qpos_adr:ball_qpos_adr + 3],
            dtype=float,
        ).copy()

    def append_sample(model, data):
        nonlocal tracked_model, tracked_data, ball_qpos_adr

        if ball_qpos_adr is None:
            joint_id = mujoco.mj_name2id(
                model,
                mujoco.mjtObj.mjOBJ_JOINT,
                "soccer_ball_joint",
            )
            if joint_id < 0:
                raise RuntimeError(
                    "soccer_ball_joint が見つかりません。"
                    "scene_soccer.xmlを確認してください。"
                )

            ball_qpos_adr = int(model.jnt_qposadr[joint_id])

        tracked_model = model
        tracked_data = data

        ball_xyz = get_ball_position(data)
        robot_xyz = np.asarray(data.qpos[:3], dtype=float)

        trajectory.append({
            "time": float(data.time),
            "ball_x": float(ball_xyz[0]),
            "ball_y": float(ball_xyz[1]),
            "ball_z": float(ball_xyz[2]),
            "robot_x": float(robot_xyz[0]),
            "robot_y": float(robot_xyz[1]),
            "robot_z": float(robot_xyz[2]),
        })

        if len(trajectory) < 2:
            return

        prev = trajectory[-2]
        curr = trajectory[-1]

        x0 = prev["ball_x"]
        x1 = curr["ball_x"]

        # X=GOAL_X を前方に通過した瞬間を補間
        if not (x0 < GOAL_X <= x1):
            return

        alpha = (GOAL_X - x0) / (x1 - x0)
        crossing_y = (
            prev["ball_y"]
            + alpha * (curr["ball_y"] - prev["ball_y"])
        )
        crossing_z = (
            prev["ball_z"]
            + alpha * (curr["ball_z"] - prev["ball_z"])
        )
        crossing_time = (
            prev["time"]
            + alpha * (curr["time"] - prev["time"])
        )

        valid = (
            abs(crossing_y) < max_abs_y
            and BALL_RADIUS <= crossing_z < max_z
        )

        goal_events.append({
            "time": crossing_time,
            "y": crossing_y,
            "z": crossing_z,
            "inside_opening": bool(valid),
        })

    def monitored_step(model, data, *args, **kwargs):
        # 最初の物理ステップ前の位置を記録
        if not trajectory:
            append_sample(model, data)

        result = original_step(model, data, *args, **kwargs)

        append_sample(model, data)
        return result

    mujoco.mj_step = monitored_step

    script = Path(__file__).with_name("record_soccer.py")
    if not script.is_file():
        raise FileNotFoundError(script)

    original_argv = sys.argv[:]

    sys.argv = [
        str(script),
        "--model", args.model,
        "--output", str(output),
        "--seconds", str(args.seconds),
    ]

    try:
        runpy.run_path(str(script), run_name="__main__")
    finally:
        mujoco.mj_step = original_step
        sys.argv = original_argv

    if len(trajectory) < 2:
        raise RuntimeError(
            "物理ステップを記録できませんでした。"
            "record_soccer.pyがmujoco.mj_stepを"
            "使用しているか確認してください。"
        )

    csv_path = result_dir / "ball_trajectory.csv"
    with csv_path.open("w", newline="") as f:
        writer = csv.DictWriter(
            f,
            fieldnames=list(trajectory[0].keys()),
        )
        writer.writeheader()
        writer.writerows(trajectory)

    positions = np.array([
        [r["ball_x"], r["ball_y"]]
        for r in trajectory
    ])

    times = np.array([r["time"] for r in trajectory])

    segments = np.diff(positions, axis=0)
    step_distances = np.linalg.norm(segments, axis=1)
    delta_times = np.diff(times)

    speeds = np.divide(
        step_distances,
        delta_times,
        out=np.zeros_like(step_distances),
        where=delta_times > 0,
    )

    delta = positions[-1] - positions[0]

    successful_goals = [
        event
        for event in goal_events
        if event["inside_opening"]
    ]

    metrics = {
        "model": args.model,
        "video": str(output),
        "simulation_seconds": float(times[-1] - times[0]),
        "samples": len(trajectory),
        "ball_delta_x_m": float(delta[0]),
        "ball_delta_y_m": float(delta[1]),
        "ball_displacement_xy_m": float(
            np.linalg.norm(delta)
        ),
        "ball_path_length_xy_m": float(
            step_distances.sum()
        ),
        "ball_max_speed_xy_mps": float(
            speeds.max()
        ),
        "goal_scored": bool(successful_goals),
        "goal_time_s": (
            float(successful_goals[0]["time"])
            if successful_goals else None
        ),
        "goal_crossings": goal_events,
        "goal_definition": {
            "plane_x": GOAL_X,
            "maximum_abs_ball_center_y": max_abs_y,
            "minimum_ball_center_z": BALL_RADIUS,
            "maximum_ball_center_z": max_z,
        },
    }

    json_path = result_dir / "metrics.json"
    json_path.write_text(
        json.dumps(metrics, indent=2, ensure_ascii=False)
    )

    print("\n========== SOCCER EVALUATION ==========")
    print(f"Ball Delta X: {metrics['ball_delta_x_m']:.4f} m")
    print(f"Ball Delta Y: {metrics['ball_delta_y_m']:.4f} m")
    print(
        "Ball displacement: "
        f"{metrics['ball_displacement_xy_m']:.4f} m"
    )
    print(
        "Ball path length: "
        f"{metrics['ball_path_length_xy_m']:.4f} m"
    )
    print(
        "Ball max speed: "
        f"{metrics['ball_max_speed_xy_mps']:.4f} m/s"
    )
    print(f"GOAL SCORED: {metrics['goal_scored']}")
    print(f"Goal time: {metrics['goal_time_s']}")
    print("Trajectory:", csv_path)
    print("Metrics:", json_path)


if __name__ == "__main__":
    main()

In [ ]:
%cd /content/Open_Duck_Playground

!MUJOCO_GL=egl uv run python eval_soccer.py \
    --model /content/checkpoints/joystick_150m_v1/2026_10_09_075517_140574720.onnx \
    --output /content/joystick_videos/soccer_evaluated.mp4 \
    --seconds 10 \
    --evaluation-dir /content/joystick_videos/soccer_evaluation

In [ ]:
from pathlib import Path
import shutil

root = Path("/content/Open_Duck_Playground")
backup = Path("/content/drive/MyDrive/OpenDuckRL/soccer_140m")
eval_dir = Path("/content/joystick_videos/soccer_evaluation")

required = {
    backup / "source/eval_soccer.py": root / "eval_soccer.py",
    backup / "evaluation/ball_trajectory.csv":
        eval_dir / "ball_trajectory.csv",
    backup / "evaluation/metrics.json":
        eval_dir / "metrics.json",
    backup / "videos/soccer_evaluated.mp4":
        Path("/content/joystick_videos/soccer_evaluated.mp4"),
}

for dst, src in required.items():
    if not src.is_file():
        raise FileNotFoundError(src)

for dst, src in required.items():
    dst.parent.mkdir(parents=True, exist_ok=True)
    shutil.copy2(src, dst)
    print("Saved:", dst)

print("\nEvaluation backup completed")

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt

path = (
    "/content/joystick_videos/"
    "soccer_evaluation/ball_trajectory.csv"
)

df = pd.read_csv(path)

plt.figure(figsize=(9, 5))

plt.plot(
    df["robot_x"], df["robot_y"],
    label="Robot", linewidth=2
)
plt.plot(
    df["ball_x"], df["ball_y"],
    label="Ball", linewidth=2
)

plt.scatter(
    df["ball_x"].iloc[0],
    df["ball_y"].iloc[0],
    marker="o", label="Ball start"
)

plt.axvline(
    2.5, linestyle="--",
    label="Goal line (X=2.5)"
)

plt.xlabel("X [m]")
plt.ylabel("Y [m]")
plt.title("Robot and Ball Trajectory")
plt.axis("equal")
plt.grid(True)
plt.legend()
plt.show()

In [ ]:
from pathlib import Path

backup = Path("/content/drive/MyDrive/OpenDuckRL/soccer_140m")

if not backup.exists():
    print("フォルダが見つかりません。Google Driveのマウントを確認してください。")
else:
    print(f"Backup: {backup}\n")

    for path in sorted(backup.rglob("*")):
        relative = path.relative_to(backup)

        if path.is_file():
            size_mb = path.stat().st_size / (1024**2)
            print(f"[FILE] {relative} ({size_mb:.2f} MB)")
        else:
            print(f"[DIR]  {relative}/")

In [ ]:
from pathlib import Path

backup = Path("/content/drive/MyDrive/OpenDuckRL/soccer_140m")

required = {
    "歩行ONNX": "model/walk_140m.onnx",
    "サッカーXML": "source/scene_soccer.xml",
    "追従カメラ": "source/record_joystick_follow.py",
    "サッカー録画": "source/record_soccer.py",
    "評価コード": "source/eval_soccer.py",
    "軌跡CSV": "evaluation/ball_trajectory.csv",
    "評価JSON": "evaluation/metrics.json",
    "バックアップ記録": "manifest.json",
}

for name, relative in required.items():
    path = backup / relative
    status = "OK" if path.is_file() and path.stat().st_size > 0 else "NG"
    print(f"{status:2}  {name}: {relative}")

# Orbaxチェックポイント確認
checkpoints = list((backup / "model").glob("*_140574720"))

if checkpoints:
    for cp in checkpoints:
        print(f"{'OK' if cp.is_dir() else 'NG'}  Orbax: {cp.name}")
else:
    print("NG  Orbaxチェックポイントが見つかりません")

In [ ]:
from google.colab import drive
drive.mount("/content/drive")

In [ ]:
%cd /content

!git clone https://github.com/apirrone/Open_Duck_Playground.git
%cd /content/Open_Duck_Playground
!uv sync

In [ ]:
from google.colab import drive
from pathlib import Path
import shutil

drive.mount("/content/drive")

root = Path("/content/Open_Duck_Playground")
backup = Path("/content/drive/MyDrive/OpenDuckRL/soccer_140m")

files = {
    backup / "model/walk_140m.onnx":
        root / "walk_140m.onnx",
    backup / "source/record_joystick_follow.py":
        root / "record_joystick_follow.py",
}

for src, dst in files.items():
    if not src.is_file():
        raise FileNotFoundError(src)

    dst.parent.mkdir(parents=True, exist_ok=True)
    shutil.copy2(src, dst)
    print("Restored:", dst.name)

In [ ]:
%%writefile /content/Open_Duck_Playground/create_delivery_scene.py
from pathlib import Path
import xml.etree.ElementTree as ET

import mujoco
import numpy as np


# ==========================================
# 1. Paths
# ==========================================

ROOT = Path("/content/Open_Duck_Playground")
XML_DIR = ROOT / "playground/open_duck_mini_v2/xmls"

SOURCE_XML = XML_DIR / "scene_flat_terrain.xml"
OUTPUT_XML = XML_DIR / "scene_delivery.xml"

# ==========================================
# 2. Physical parameters
# ==========================================

# Tray: 28 cm x 24 cm, thickness 1.6 cm
TRAY_HALF_SIZE = (0.14, 0.12, 0.008)
TRAY_MASS_KG = 0.08

# Relative to robot floating-base body
TRAY_LOCAL_POS = (0.0, 0.0, 0.16)

# Cup: diameter 6 cm, height 8 cm
CUP_RADIUS = 0.03
CUP_HALF_HEIGHT = 0.04
CUP_MASS_KG = 0.18

# Small initial clearance above the tray
CUP_CLEARANCE = 0.003


def vec_to_str(values):
    return " ".join(f"{float(v):.8f}" for v in values)


def find_robot_base(worldbody):
    """Find the robot's floating-base body."""

    candidates = []

    for body in worldbody.iter("body"):
        freejoint = body.find("freejoint") is not None

        free_joint = any(
            joint.get("type") == "free"
            for joint in body.findall("joint")
        )

        if freejoint or free_joint:
            candidates.append(body)

    if len(candidates) != 1:
        raise RuntimeError(
            "Expected one robot floating base, "
            f"found {len(candidates)}: "
            f"{[body.get('name') for body in candidates]}"
        )

    return candidates[0]


def main():
    # ======================================
    # 3. Load original scene
    # ======================================

    if not SOURCE_XML.is_file():
        raise FileNotFoundError(SOURCE_XML)

    tree = ET.parse(SOURCE_XML)
    xml_root = tree.getroot()

    worldbody = xml_root.find("worldbody")

    if worldbody is None:
        raise RuntimeError("worldbody not found")

    robot_base = find_robot_base(worldbody)

    print("Robot base body:", robot_base.get("name"))

    # ======================================
    # 4. Add fixed tray to robot
    # ======================================

    tray_body = ET.SubElement(
        robot_base,
        "body",
        {
            "name": "delivery_tray",
            "pos": vec_to_str(TRAY_LOCAL_POS),
        },
    )

    ET.SubElement(
        tray_body,
        "geom",
        {
            "name": "delivery_tray_surface",
            "type": "box",
            "size": vec_to_str(TRAY_HALF_SIZE),
            "mass": str(TRAY_MASS_KG),
            "friction": "1.0 0.005 0.0001",
            "rgba": "0.65 0.42 0.23 1",
            "contype": "1",
            "conaffinity": "1",
        },
    )

    # ======================================
    # 5. Calculate initial tray position
    # ======================================

    # Temporarily save scene containing tray only.
    tree.write(
        OUTPUT_XML,
        encoding="unicode",
        xml_declaration=True,
    )

    temp_model = mujoco.MjModel.from_xml_path(
        str(OUTPUT_XML)
    )
    temp_data = mujoco.MjData(temp_model)

    mujoco.mj_forward(temp_model, temp_data)

    tray_id = mujoco.mj_name2id(
        temp_model,
        mujoco.mjtObj.mjOBJ_GEOM,
        "delivery_tray_surface",
    )

    if tray_id < 0:
        raise RuntimeError("Tray geom not found")

    tray_center = temp_data.geom_xpos[tray_id].copy()

    tray_rotation = (
        temp_data.geom_xmat[tray_id]
        .reshape(3, 3)
        .copy()
    )

    # Tray top position in world coordinates
    tray_top = (
        tray_center
        + tray_rotation[:, 2] * TRAY_HALF_SIZE[2]
    )

    # Cup center is above tray top
    cup_center = (
        tray_top
        + np.array(
            [0.0, 0.0, CUP_HALF_HEIGHT + CUP_CLEARANCE]
        )
    )

    print("Tray center:", tray_center)
    print("Tray top:", tray_top)
    print("Cup center:", cup_center)

    # ======================================
    # 6. Add free-moving cup
    # ======================================

    cup_body = ET.SubElement(
        worldbody,
        "body",
        {
            "name": "yunomi",
            "pos": vec_to_str(cup_center),
        },
    )

    ET.SubElement(
        cup_body,
        "freejoint",
        {
            "name": "yunomi_freejoint",
        },
    )

    ET.SubElement(
        cup_body,
        "geom",
        {
            "name": "yunomi_collision",
            "type": "cylinder",
            "size": vec_to_str(
                (CUP_RADIUS, CUP_HALF_HEIGHT)
            ),
            "mass": str(CUP_MASS_KG),
            "friction": "0.9 0.005 0.0001",
            "rgba": "0.92 0.90 0.82 1",
            "contype": "1",
            "conaffinity": "1",
        },
    )

    # ======================================
    # 7. Write final XML
    # ======================================

    tree.write(
        OUTPUT_XML,
        encoding="unicode",
        xml_declaration=True,
    )

    # ======================================
    # 8. Validate MuJoCo model
    # ======================================

    model = mujoco.MjModel.from_xml_path(
        str(OUTPUT_XML)
    )

    data = mujoco.MjData(model)
    mujoco.mj_forward(model, data)

    if model.nu != 14:
        raise RuntimeError(
            f"Expected 14 actuators, got {model.nu}"
        )

    cup_body_id = mujoco.mj_name2id(
        model,
        mujoco.mjtObj.mjOBJ_BODY,
        "yunomi",
    )

    tray_geom_id = mujoco.mj_name2id(
        model,
        mujoco.mjtObj.mjOBJ_GEOM,
        "delivery_tray_surface",
    )

    cup_joint_id = mujoco.mj_name2id(
        model,
        mujoco.mjtObj.mjOBJ_JOINT,
        "yunomi_freejoint",
    )

    if min(
        cup_body_id,
        tray_geom_id,
        cup_joint_id,
    ) < 0:
        raise RuntimeError(
            "Required tray/cup elements not found"
        )

    if model.jnt_type[cup_joint_id] != (
        mujoco.mjtJoint.mjJNT_FREE
    ):
        raise RuntimeError(
            "Cup joint is not a free joint"
        )

    cup_position = data.xpos[cup_body_id].copy()

    print()
    print("========== DELIVERY SCENE ==========")
    print("XML:", OUTPUT_XML)
    print("Bodies:", model.nbody)
    print("Geoms:", model.ngeom)
    print("Actuators:", model.nu)
    print("Cup initial position:", cup_position)
    print("Cup freejoint: OK")
    print("MuJoCo XML validation: OK")


if __name__ == "__main__":
    main()

In [ ]:
%cd /content/Open_Duck_Playground

!uv run python create_delivery_scene.py

In [ ]:
from pathlib import Path

root = Path("/content/Open_Duck_Playground")

script = root / "create_delivery_scene.py"
xml = root / "playground/open_duck_mini_v2/xmls/scene_delivery.xml"

print("Pythonスクリプト:", script.exists())
print("配膳用XML:", xml.exists())

if xml.exists():
    print("XMLサイズ:", xml.stat().st_size, "bytes")

In [ ]:
%cd /content/Open_Duck_Playground

!uv run python -c "import mujoco; m=mujoco.MjModel.from_xml_path('playground/open_duck_mini_v2/xmls/scene_flat_terrain.xml'); print([(m.body(i).name, int(m.body_jntnum[i])) for i in range(m.nbody)])"

In [ ]:
from pathlib import Path

root = Path("/content/Open_Duck_Playground")
script = root / "export_expanded_scene.py"

script.write_text('''
from pathlib import Path
import mujoco

xml_dir = Path(
    "/content/Open_Duck_Playground/playground/open_duck_mini_v2/xmls"
)

src = xml_dir / "scene_flat_terrain.xml"
dst = xml_dir / "scene_flat_expanded.xml"

model = mujoco.MjModel.from_xml_path(str(src))
mujoco.mj_saveLastXML(str(dst), model)

print("Expanded XML:", dst)
''')

print("Created:", script)

In [ ]:
%cd /content/Open_Duck_Playground

!uv run python export_expanded_scene.py

In [ ]:
%cd /content/Open_Duck_Playground

!uv run python -c "import mujoco; m=mujoco.MjModel.from_xml_path('playground/open_duck_mini_v2/xmls/scene_flat_expanded.xml'); print('Bodies:', m.nbody, 'Actuators:', m.nu)"

In [ ]:
%%writefile /content/Open_Duck_Playground/create_delivery_scene_v2.py
from pathlib import Path
import xml.etree.ElementTree as ET

import mujoco
import numpy as np

ROOT = Path("/content/Open_Duck_Playground")
XML_DIR = ROOT / "playground/open_duck_mini_v2/xmls"

SRC = XML_DIR / "scene_flat_expanded.xml"
DST = XML_DIR / "scene_delivery.xml"

# トレーの設定
TRAY_POS = [0.0, 0.0, 0.16]
TRAY_SIZE = [0.14, 0.12, 0.008]
TRAY_MASS = 0.08

# 湯飲みの設定
CUP_RADIUS = 0.03
CUP_HALF_HEIGHT = 0.04
CUP_MASS = 0.18
CUP_CLEARANCE = 0.003


def vec(values):
    return " ".join(f"{float(v):.8f}" for v in values)


def main():
    if not SRC.is_file():
        raise FileNotFoundError(SRC)

    tree = ET.parse(SRC)
    xml_root = tree.getroot()

    worldbody = xml_root.find("worldbody")
    if worldbody is None:
        raise RuntimeError("worldbody not found")

    # 展開済みXMLからbaseを検索
    robot_base = worldbody.find(".//body[@name='base']")
    if robot_base is None:
        raise RuntimeError("Robot body 'base' not found")

    print("Robot base:", robot_base.get("name"))

    # トレーをbaseに固定
    tray_body = ET.SubElement(
        robot_base,
        "body",
        {
            "name": "delivery_tray",
            "pos": vec(TRAY_POS),
        },
    )

    ET.SubElement(
        tray_body,
        "geom",
        {
            "name": "delivery_tray_surface",
            "type": "box",
            "size": vec(TRAY_SIZE),
            "mass": str(TRAY_MASS),
            "friction": "1.0 0.005 0.0001",
            "rgba": "0.65 0.42 0.23 1",
        },
    )

    # トレー追加後のXMLを一時的に保存
    tree.write(DST, encoding="unicode", xml_declaration=True)

    model = mujoco.MjModel.from_xml_path(str(DST))
    data = mujoco.MjData(model)
    mujoco.mj_forward(model, data)

    tray_id = mujoco.mj_name2id(
        model,
        mujoco.mjtObj.mjOBJ_GEOM,
        "delivery_tray_surface",
    )

    tray_center = data.geom_xpos[tray_id].copy()
    tray_rot = data.geom_xmat[tray_id].reshape(3, 3).copy()

    # トレー上面の中心
    tray_top = tray_center + tray_rot[:, 2] * TRAY_SIZE[2]

    # 湯飲みの初期位置
    cup_center = (
        tray_top
        + np.array([0.0, 0.0, CUP_HALF_HEIGHT + CUP_CLEARANCE])
    )

    # 自由に動く湯飲み
    cup = ET.SubElement(
        worldbody,
        "body",
        {
            "name": "yunomi",
            "pos": vec(cup_center),
        },
    )

    ET.SubElement(
        cup,
        "freejoint",
        {"name": "yunomi_freejoint"},
    )

    ET.SubElement(
        cup,
        "geom",
        {
            "name": "yunomi_collision",
            "type": "cylinder",
            "size": vec([CUP_RADIUS, CUP_HALF_HEIGHT]),
            "mass": str(CUP_MASS),
            "friction": "0.9 0.005 0.0001",
            "rgba": "0.92 0.90 0.82 1",
        },
    )

    # 完成版XMLを保存
    tree.write(DST, encoding="unicode", xml_declaration=True)

    # 最終検証
    final_model = mujoco.MjModel.from_xml_path(str(DST))
    final_data = mujoco.MjData(final_model)
    mujoco.mj_forward(final_model, final_data)

    assert final_model.nu == 14, "Actuator count changed"

    cup_joint_id = mujoco.mj_name2id(
        final_model,
        mujoco.mjtObj.mjOBJ_JOINT,
        "yunomi_freejoint",
    )
    assert cup_joint_id >= 0, "Cup freejoint not found"

    print()
    print("========== DELIVERY SCENE ==========")
    print("Created:", DST)
    print("Bodies:", final_model.nbody)
    print("Geoms:", final_model.ngeom)
    print("Actuators:", final_model.nu)
    print("Tray center:", tray_center)
    print("Cup center:", cup_center)
    print("Cup freejoint: OK")
    print("MuJoCo XML validation: OK")


if __name__ == "__main__":
    main()

In [ ]:
%cd /content/Open_Duck_Playground

!uv run python create_delivery_scene_v2.py

In [ ]:
from pathlib import Path

root = Path("/content/Open_Duck_Playground")

src = root / "record_joystick_follow.py"
dst = root / "record_delivery.py"

code = src.read_text()

old = "scene_flat_terrain.xml"
new = "scene_delivery.xml"

if old not in code:
    raise RuntimeError("scene_flat_terrain.xml が見つかりません")

dst.write_text(code.replace(old, new))

print("Created:", dst)

In [ ]:
%cd /content/Open_Duck_Playground

!uv pip install --python .venv/bin/python "imageio[ffmpeg]"

In [ ]:
%cd /content/Open_Duck_Playground

!MUJOCO_GL=egl uv run python record_delivery.py \
    --model /content/Open_Duck_Playground/walk_140m.onnx \
    --output /content/delivery_140m_test.mp4 \
    --seconds 10

In [ ]:
%cd /content/Open_Duck_Playground

!uv add "imageio[ffmpeg]"

In [ ]:
from IPython.display import Video, display

display(Video("/content/delivery_140m_test.mp4", embed=True, width=800))

In [ ]:
from pathlib import Path

path = Path(
    "/content/Open_Duck_Playground/create_delivery_scene_v2.py"
)

code = path.read_text()

old = """    # トレーをbaseに固定
    tray_body = ET.SubElement("""

new = """    # 頭部の位置を取得してトレーの高さを決定
    original_model = mujoco.MjModel.from_xml_path(str(SRC))
    original_data = mujoco.MjData(original_model)
    mujoco.mj_forward(original_model, original_data)

    head_id = mujoco.mj_name2id(
        original_model,
        mujoco.mjtObj.mjOBJ_BODY,
        "head_assembly",
    )
    base_id = mujoco.mj_name2id(
        original_model,
        mujoco.mjtObj.mjOBJ_BODY,
        "base",
    )

    if head_id < 0 or base_id < 0:
        raise RuntimeError("head/base body not found")

    head_pos = original_data.xpos[head_id]
    base_pos = original_data.xpos[base_id]

    # 頭部bodyの中心より15cm上にトレー中心を配置
    desired_world_pos = head_pos + np.array([0.0, 0.0, 0.15])

    base_rot = original_data.xmat[base_id].reshape(3, 3)

    # ワールド座標からbaseのローカル座標へ変換
    tray_local_pos = base_rot.T @ (desired_world_pos - base_pos)

    print("Head position:", head_pos)
    print("Tray local position:", tray_local_pos)

    # トレーをbaseに固定
    tray_body = ET.SubElement("""

if old not in code:
    raise RuntimeError("置換対象が見つかりません")

code = code.replace(old, new)

code = code.replace(
    '"pos": vec(TRAY_POS),',
    '"pos": vec(tray_local_pos),',
)

path.write_text(code)

print("Updated:", path)

In [ ]:
%cd /content/Open_Duck_Playground

!uv run python create_delivery_scene_v2.py

In [ ]:
%cd /content/Open_Duck_Playground

!MUJOCO_GL=egl uv run python record_delivery.py \
    --model /content/Open_Duck_Playground/walk_140m.onnx \
    --output /content/delivery_140m_head_tray.mp4 \
    --seconds 10

In [ ]:
from IPython.display import Video, display

display(Video(
    "/content/delivery_140m_head_tray.mp4",
    embed=True,
    width=800,
))

In [ ]:
from pathlib import Path
import shutil

script = Path(
    "/content/Open_Duck_Playground/create_delivery_scene_v2.py"
)

if not script.is_file():
    raise FileNotFoundError(script)

# 修正前を保存
original = script.with_name("create_delivery_scene_v2_before_lower.py")
shutil.copy2(script, original)

code = script.read_text()

old = "head_pos + np.array([0.0, 0.0, 0.15])"
new = "head_pos + np.array([0.0, 0.0, 0.10])"

if code.count(old) != 1:
    raise RuntimeError(
        "トレーの高さ指定が見つかりません。"
        "create_delivery_scene_v2.py を確認してください。"
    )

script.write_text(code.replace(old, new))

print("トレーを5cm下げる設定に変更しました")
print("Backup:", original)

In [ ]:
%cd /content/Open_Duck_Playground
!uv run python create_delivery_scene_v2.py

In [ ]:
!MUJOCO_GL=egl uv run python record_delivery.py \
    --model /content/Open_Duck_Playground/walk_140m.onnx \
    --output /content/delivery_140m_lower_tray.mp4 \
    --seconds 10

In [ ]:
from IPython.display import Video, display

display(Video(
    "/content/delivery_140m_lower_tray.mp4",
    embed=True,
    width=800,
))

In [ ]:
%%writefile /content/Open_Duck_Playground/playground/open_duck_mini_v2/delivery_rewards.py
"""Rewards and termination signals for cup delivery PPO."""

import jax.numpy as jnp


def delivery_reward(
    cup_xy_local,
    cup_vel_xy_local,
    cup_up_dot,
    tray_angular_velocity,
    residual_action,
    previous_residual_action,
    robot_alive,
    dt=0.02,
):
    """
    All vectors must be JAX arrays.

    cup_xy_local:
        Cup center relative to tray, expressed in tray coordinates.
    cup_vel_xy_local:
        Relative XY velocity in tray coordinates.
    cup_up_dot:
        Dot product of cup up-axis and world up-axis.
        1 means upright.
    tray_angular_velocity:
        Tray angular velocity in world coordinates.
    residual_action:
        Residual PPO action, before scaling.
    robot_alive:
        Boolean indicating that the robot has not fallen.
    """

    distance = jnp.linalg.norm(cup_xy_local)
    slip_speed = jnp.linalg.norm(cup_vel_xy_local)
    angular_speed = jnp.linalg.norm(tray_angular_velocity)

    # Cup stays close to tray center
    center_reward = jnp.exp(
        -(distance / 0.05) ** 2
    )

    # Cup remains upright
    upright_reward = jnp.clip(
        cup_up_dot, -1.0, 1.0
    ) ** 2

    # No sliding
    slip_penalty = slip_speed ** 2

    # Reduce tray oscillation
    vibration_penalty = angular_speed ** 2

    # Avoid unnecessary correction
    action_penalty = jnp.sum(residual_action ** 2)

    # Reduce jerky correction
    smoothness_penalty = jnp.sum(
        (residual_action - previous_residual_action) ** 2
    )

    # Geometry of 28 x 24 cm tray.
    # Use margin for the cup radius (3 cm).
    inside_tray = (
        (jnp.abs(cup_xy_local[0]) < 0.11)
        & (jnp.abs(cup_xy_local[1]) < 0.09)
    )

    # Tilt limit: 45 degrees
    upright_enough = cup_up_dot > 0.7071

    cup_safe = inside_tray & upright_enough
    failed = (~cup_safe) | (~robot_alive)

    # Reward and penalties per second
    reward_rate = (
        3.0 * center_reward
        + 3.0 * upright_reward
        - 2.0 * slip_penalty
        - 0.2 * vibration_penalty
        - 0.02 * action_penalty
        - 0.05 * smoothness_penalty
    )

    # Convert per-second rewards to step rewards
    reward = reward_rate * dt

    # Apply one-time termination penalty
    reward = reward - 20.0 * failed.astype(jnp.float32)

    metrics = {
        "cup_center_distance": distance,
        "cup_slip_speed": slip_speed,
        "cup_up_dot": cup_up_dot,
        "tray_angular_speed": angular_speed,
        "cup_safe": cup_safe.astype(jnp.float32),
    }

    return reward, failed, metrics

In [ ]:
%cd /content/Open_Duck_Playground

In [ ]:
import subprocess

result = subprocess.run(
    [
        "uv", "run", "python", "-c",
        """
import mujoco

path = "playground/open_duck_mini_v2/xmls/scene_delivery.xml"
model = mujoco.MjModel.from_xml_path(path)

print("nq:", model.nq)
print("nv:", model.nv)
print("nu:", model.nu)
print("keyframes:", model.nkey)

for i in range(model.njnt):
    print(
        "joint:",
        model.joint(i).name,
        "type:",
        int(model.jnt_type[i]),
        "qposadr:",
        int(model.jnt_qposadr[i])
    )

for name in ["base", "delivery_tray", "yunomi"]:
    print("body:", name, model.body(name).id)

for name in ["delivery_tray_surface", "yunomi_collision"]:
    print("geom:", name, model.geom(name).id)

if model.nkey:
    print("home keyframe:", model.keyframe("home").qpos.shape)
"""
    ],
    cwd="/content/Open_Duck_Playground",
    capture_output=True,
    text=True,
)

print(result.stdout)
if result.returncode:
    print(result.stderr)

In [ ]:
%%writefile /content/Open_Duck_Playground/create_delivery_fluid.py
from pathlib import Path
import math
import xml.etree.ElementTree as ET
import mujoco

XML_DIR = Path(
    "/content/Open_Duck_Playground/"
    "playground/open_duck_mini_v2/xmls"
)

SRC = XML_DIR / "scene_delivery.xml"
DST = XML_DIR / "scene_delivery_fluid.xml"

# Cup dimensions [m]
R = 0.030
H = 0.040
WALL = 0.003
BOTTOM = 0.004

# Masses [kg]
CERAMIC_MASS = 0.100
TEA_MASS = 0.100

# Sloshing proxy
SLOSH_LIMIT = 0.015
SLOSH_STIFFNESS = 10.0
SLOSH_DAMPING = 0.35

N_WALL = 16


def fmt(values):
    return " ".join(f"{float(x):.8f}" for x in values)


def main():
    if not SRC.exists():
        raise FileNotFoundError(SRC)

    tree = ET.parse(SRC)
    root = tree.getroot()
    world = root.find("worldbody")

    cup = world.find("./body[@name='yunomi']")
    if cup is None:
        raise RuntimeError("yunomi body not found")

    # Preserve the existing cup pose and freejoint.
    original_pos = cup.get("pos")

    for child in list(cup):
        if child.tag != "freejoint":
            cup.remove(child)

    # Bottom of the ceramic cup
    ET.SubElement(
        cup, "geom",
        {
            "name": "yunomi_bottom",
            "type": "cylinder",
            "pos": fmt([0, 0, -H + BOTTOM / 2]),
            "size": fmt([R, BOTTOM / 2]),
            "mass": str(CERAMIC_MASS * 0.25),
            "friction": "0.9 0.005 0.0001",
            "rgba": "0.90 0.88 0.80 1",
        }
    )

    # 16 wall segments forming a hollow cup.
    for i in range(N_WALL):
        theta = 2 * math.pi * i / N_WALL

        radius = R - WALL / 2
        x = radius * math.cos(theta)
        y = radius * math.sin(theta)

        ET.SubElement(
            cup, "geom",
            {
                "name": f"yunomi_wall_{i:02d}",
                "type": "box",
                "pos": fmt([x, y, 0]),
                "euler": fmt([0, 0, math.degrees(theta)]),
                "size": fmt([
                    WALL / 2,
                    R * math.sin(math.pi / N_WALL) + 0.001,
                    H,
                ]),
                "mass": str(
                    CERAMIC_MASS * 0.75 / N_WALL
                ),
                "friction": "0.9 0.005 0.0001",
                "rgba": "0.90 0.88 0.80 1",
            }
        )

    # Internal moving mass representing liquid slosh.
    tea = ET.SubElement(
        cup, "body",
        {
            "name": "tea_slosh",
            "pos": fmt([0, 0, -0.015]),
        }
    )

    for axis, name in [
        ([1, 0, 0], "x"),
        ([0, 1, 0], "y"),
    ]:
        ET.SubElement(
            tea, "joint",
            {
                "name": f"tea_slosh_{name}",
                "type": "slide",
                "axis": fmt(axis),
                "limited": "true",
                "range": fmt([
                    -SLOSH_LIMIT,
                    SLOSH_LIMIT,
                ]),
                "stiffness": str(SLOSH_STIFFNESS),
                "damping": str(SLOSH_DAMPING),
                "armature": "0",
            }
        )

    # A moving ellipsoid: visual proxy for tea.
    # No collisions: slosh is limited by joint ranges.
    ET.SubElement(
        tea, "geom",
        {
            "name": "tea_proxy",
            "type": "ellipsoid",
            "size": "0.020 0.020 0.016",
            "mass": str(TEA_MASS),
            "contype": "0",
            "conaffinity": "0",
            "rgba": "0.30 0.58 0.28 0.65",
        }
    )

    # The original home keyframe has two fewer joint
    # coordinates/velocities. Extend it by two zeros.
    keyframe = root.find("keyframe")

    if keyframe is not None:
        for key in keyframe.findall("key"):
            for attr in ("qpos", "qvel"):
                value = key.get(attr)
                if value is not None:
                    key.set(attr, value + " 0 0")

    tree.write(
        DST,
        encoding="unicode",
        xml_declaration=True,
    )

    model = mujoco.MjModel.from_xml_path(str(DST))
    data = mujoco.MjData(model)
    mujoco.mj_forward(model, data)

    assert model.nu == 14
    assert model.nq == 30, f"Unexpected nq: {model.nq}"
    assert model.nv == 28, f"Unexpected nv: {model.nv}"

    print("Source:", SRC)
    print("Created:", DST)
    print("Cup position:", original_pos)
    print("Actuators:", model.nu)
    print("nq:", model.nq)
    print("nv:", model.nv)
    print("Slosh joints: 2")
    print("Fluid proxy mass:", TEA_MASS, "kg")
    print("MuJoCo XML validation: OK")


if __name__ == "__main__":
    main()

In [ ]:
%cd /content/Open_Duck_Playground
!uv run python create_delivery_fluid.py

In [ ]:
from pathlib import Path

root = Path("/content/Open_Duck_Playground")

src = root / "record_delivery.py"
dst = root / "record_delivery_fluid.py"

code = src.read_text()

old = "scene_delivery.xml"
new = "scene_delivery_fluid.xml"

if old not in code:
    raise RuntimeError("XML指定が見つかりません")

dst.write_text(code.replace(old, new))
print("Created:", dst)

In [ ]:
%cd /content/Open_Duck_Playground

!MUJOCO_GL=egl uv run python record_delivery_fluid.py \
    --model /content/Open_Duck_Playground/walk_140m.onnx \
    --output /content/delivery_fluid_140m.mp4 \
    --seconds 10

In [ ]:
from IPython.display import Video, display

display(Video(
    "/content/delivery_fluid_140m.mp4",
    embed=True,
    width=800,
))

In [ ]:
%%writefile /content/Open_Duck_Playground/create_delivery_tea_v2.py
from pathlib import Path
import math
import xml.etree.ElementTree as ET

import mujoco
import numpy as np


# -----------------------------------
# Paths
# -----------------------------------

XML_DIR = Path(
    "/content/Open_Duck_Playground/"
    "playground/open_duck_mini_v2/xmls"
)

SOURCE = XML_DIR / "scene_delivery.xml"
OUTPUT = XML_DIR / "scene_delivery_tea_v2.xml"


# -----------------------------------
# Cup dimensions and masses
# -----------------------------------

CUP_RADIUS = 0.030
CUP_HALF_HEIGHT = 0.040
WALL_THICKNESS = 0.003
BOTTOM_THICKNESS = 0.004

CUP_MASS = 0.100
TEA_MASS = 0.100

N_WALLS = 16

# Tea surface inside the cup.
# Cup top: +0.040
# Cup bottom: -0.040
TEA_LEVEL_Z = 0.008
TEA_SURFACE_RADIUS = 0.024
TEA_SURFACE_HALF_THICKNESS = 0.001

# Passive sloshing joints
MAX_SLOSH_ANGLE = math.radians(12.0)
SLOSH_STIFFNESS = 0.020
SLOSH_DAMPING = 0.004


def vec(values):
    return " ".join(f"{float(v):.8f}" for v in values)


def add_geom(parent, **attrs):
    return ET.SubElement(
        parent,
        "geom",
        {k: str(v) for k, v in attrs.items()},
    )


def main():
    if not SOURCE.is_file():
        raise FileNotFoundError(SOURCE)

    tree = ET.parse(SOURCE)
    root = tree.getroot()
    world = root.find("worldbody")

    if world is None:
        raise RuntimeError("worldbody not found")

    cup = world.find("./body[@name='yunomi']")

    if cup is None:
        raise RuntimeError("yunomi body not found")

    # Keep the existing freejoint and cup pose.
    joints = [
        e for e in cup
        if e.tag == "freejoint"
    ]

    if len(joints) != 1:
        raise RuntimeError(
            "Expected exactly one yunomi freejoint"
        )

    # Remove the old solid cup geometry.
    # Keep the cup freejoint.
    for child in list(cup):
        if child.tag != "freejoint":
            cup.remove(child)

    # -----------------------------------
    # Ceramic bottom
    # -----------------------------------

    add_geom(
        cup,
        name="yunomi_bottom_v2",
        type="cylinder",
        pos=vec([
            0,
            0,
            -CUP_HALF_HEIGHT + BOTTOM_THICKNESS / 2,
        ]),
        size=vec([
            CUP_RADIUS,
            BOTTOM_THICKNESS / 2,
        ]),
        mass=CUP_MASS * 0.25,
        friction="0.9 0.005 0.0001",
        rgba="0.94 0.90 0.80 1",
    )

    # -----------------------------------
    # Ceramic walls
    # -----------------------------------

    wall_radius = CUP_RADIUS - WALL_THICKNESS / 2
    wall_length = (
        CUP_RADIUS * math.sin(math.pi / N_WALLS)
        + 0.001
    )

    for i in range(N_WALLS):
        theta = 2 * math.pi * i / N_WALLS

        x = wall_radius * math.cos(theta)
        y = wall_radius * math.sin(theta)

        add_geom(
            cup,
            name=f"yunomi_wall_v2_{i:02d}",
            type="box",
            pos=vec([x, y, 0]),
            euler=vec([0, 0, math.degrees(theta)]),
            size=vec([
                WALL_THICKNESS / 2,
                wall_length,
                CUP_HALF_HEIGHT,
            ]),
            mass=CUP_MASS * 0.75 / N_WALLS,
            friction="0.9 0.005 0.0001",
            rgba="0.94 0.90 0.80 0.75",
        )

    # -----------------------------------
    # Sloshing model
    # -----------------------------------

    # This body is inside the cup.
    # Its rotation represents liquid slosh.
    tea = ET.SubElement(
        cup,
        "body",
        {
            "name": "tea_surface_body",
            "pos": vec([0, 0, TEA_LEVEL_Z]),
        },
    )

    # Two passive rotational joints.
    for name, axis in [
        ("tea_slosh_x", [1, 0, 0]),
        ("tea_slosh_y", [0, 1, 0]),
    ]:
        ET.SubElement(
            tea,
            "joint",
            {
                "name": name,
                "type": "hinge",
                "axis": vec(axis),
                "limited": "true",
                "range": vec([
                    -MAX_SLOSH_ANGLE,
                    MAX_SLOSH_ANGLE,
                ]),
                "stiffness": str(SLOSH_STIFFNESS),
                "damping": str(SLOSH_DAMPING),
                "armature": "0",
            },
        )

    # Visual liquid surface.
    # A thin disk, not an external ellipsoid.
    add_geom(
        tea,
        name="tea_surface_visual",
        type="cylinder",
        pos="0 0 0",
        size=vec([
            TEA_SURFACE_RADIUS,
            TEA_SURFACE_HALF_THICKNESS,
        ]),
        mass="0.0001",
        contype="0",
        conaffinity="0",
        rgba="0.25 0.48 0.20 0.85",
    )

    # Internal mass representing the liquid.
    # Placed below the surface pivot to create
    # pendulum-like restoring dynamics.
    #
    # Invisible and non-colliding.
    add_geom(
        tea,
        name="tea_internal_mass",
        type="ellipsoid",
        pos="0 0 -0.033",
        size="0.012 0.012 0.012",
        mass=str(TEA_MASS),
        contype="0",
        conaffinity="0",
        rgba="0 0 0 0",
    )

    # -----------------------------------
    # Extend existing keyframes
    # -----------------------------------

    keyframe = root.find("keyframe")

    if keyframe is not None:
        for key in keyframe.findall("key"):
            for attr in ("qpos", "qvel"):
                value = key.get(attr)

                if value is not None:
                    key.set(
                        attr,
                        value.strip() + " 0 0",
                    )

    # -----------------------------------
    # Save and validate
    # -----------------------------------

    tree.write(
        OUTPUT,
        encoding="unicode",
        xml_declaration=True,
    )

    model = mujoco.MjModel.from_xml_path(
        str(OUTPUT)
    )

    data = mujoco.MjData(model)
    mujoco.mj_forward(model, data)

    assert model.nu == 14
    assert model.nq == 30, f"nq={model.nq}"
    assert model.nv == 28, f"nv={model.nv}"

    for name in (
        "yunomi_freejoint",
        "tea_slosh_x",
        "tea_slosh_y",
    ):
        assert mujoco.mj_name2id(
            model,
            mujoco.mjtObj.mjOBJ_JOINT,
            name,
        ) >= 0

    print("Created:", OUTPUT)
    print("Actuators:", model.nu)
    print("nq:", model.nq)
    print("nv:", model.nv)
    print("Cup mass:", CUP_MASS, "kg")
    print("Tea mass:", TEA_MASS, "kg")
    print("MuJoCo XML validation: OK")


if __name__ == "__main__":
    main()

In [ ]:
%cd /content/Open_Duck_Playground

!uv run python create_delivery_tea_v2.py

In [ ]:
from pathlib import Path

root = Path("/content/Open_Duck_Playground")

src = root / "record_delivery.py"
dst = root / "record_delivery_tea_v2.py"

code = src.read_text()

old = "scene_delivery.xml"
new = "scene_delivery_tea_v2.xml"

if old not in code:
    raise RuntimeError("XML reference not found")

dst.write_text(code.replace(old, new))

print("Created:", dst)

In [ ]:
%cd /content/Open_Duck_Playground

!MUJOCO_GL=egl uv run python record_delivery_tea_v2.py \
    --model /content/Open_Duck_Playground/walk_140m.onnx \
    --output /content/delivery_tea_v2.mp4 \
    --seconds 10

In [ ]:
from IPython.display import Video, display

display(Video(
    "/content/delivery_tea_v2.mp4",
    embed=True,
    width=800,
))

In [ ]:
from google.colab import drive
from pathlib import Path
from datetime import datetime
import shutil

drive.mount("/content/drive")

# 保存先
backup = Path(
    "/content/drive/MyDrive/OpenDuckRL/all_videos_20261009"
)

# これまで動画を生成したディレクトリ
search_dirs = [
    Path("/content/joystick_videos"),
    Path("/content/Open_Duck_Playground"),
    Path("/content"),
]

# Colab全体を重複して探索しないように、
# /content 以下を1回だけ検索する
video_files = []

for path in Path("/content").rglob("*"):
    if not path.is_file():
        continue

    if path.suffix.lower() not in (".mp4", ".mov", ".avi", ".webm"):
        continue

    # Google Drive内は検索対象から除外
    if "/content/drive/" in str(path):
        continue

    video_files.append(path)

video_files = sorted(set(video_files))

print(f"検出した動画: {len(video_files)} 本")

total_bytes = 0

for src in video_files:
    # 元ディレクトリ構造を保持
    relative = src.relative_to("/content")
    dst = backup / relative

    dst.parent.mkdir(parents=True, exist_ok=True)

    shutil.copy2(src, dst)

    src_size = src.stat().st_size
    dst_size = dst.stat().st_size

    if src_size != dst_size:
        raise RuntimeError(f"コピーサイズ不一致: {src}")

    total_bytes += src_size

    print(
        f"OK: {relative} "
        f"({src_size / 1024**2:.2f} MB)"
    )

print()
print("========== BACKUP COMPLETED ==========")
print("保存先:", backup)
print("保存動画数:", len(video_files))
print("合計サイズ:", round(total_bytes / 1024**2, 2), "MB")

In [ ]:
from pathlib import Path

backup = Path(
    "/content/drive/MyDrive/OpenDuckRL/all_videos_20261009"
)

videos = sorted(
    p for p in backup.rglob("*")
    if p.is_file()
    and p.suffix.lower() in (".mp4", ".mov", ".avi", ".webm")
)

for i, video in enumerate(videos, 1):
    size_mb = video.stat().st_size / 1024**2
    print(
        f"{i:02d}. {video.relative_to(backup)} "
        f"({size_mb:.2f} MB)"
    )

print(f"\n合計: {len(videos)} 本")

In [ ]:
from pathlib import Path
import shutil

root = Path("/content/Open_Duck_Playground")

backup = Path(
    "/content/drive/MyDrive/OpenDuckRL/delivery_140m"
)

files = {
    "source/scene_delivery_tea_v2.xml":
        root / "playground/open_duck_mini_v2/xmls/scene_delivery_tea_v2.xml",
    "source/create_delivery_tea_v2.py":
        root / "create_delivery_tea_v2.py",
    "source/record_delivery_tea_v2.py":
        root / "record_delivery_tea_v2.py",
    "source/scene_delivery.xml":
        root / "playground/open_duck_mini_v2/xmls/scene_delivery.xml",
}

for name, src in files.items():
    if src.is_file():
        dst = backup / name
        dst.parent.mkdir(parents=True, exist_ok=True)
        shutil.copy2(src, dst)
        print("Saved:", name)
    else:
        print("NOT FOUND:", src)

In [ ]:
from google.colab import drive
from pathlib import Path
import shutil

drive.mount("/content/drive")

root = Path("/content/Open_Duck_Playground")
xml_dir = root / "playground/open_duck_mini_v2/xmls"

backup = Path(
    "/content/drive/MyDrive/OpenDuckRL/delivery_140m/source"
)

print("=== Drive内のファイル ===")

if backup.exists():
    for p in sorted(backup.iterdir()):
        print(p.name)
else:
    print("バックアップフォルダが見つかりません")

# 復元するXML
src = backup / "scene_delivery_tea_v2.xml"
dst = xml_dir / "scene_delivery_tea_v2.xml"

if not src.is_file():
    print("\n復元元がありません:", src)
else:
    xml_dir.mkdir(parents=True, exist_ok=True)
    shutil.copy2(src, dst)
    print("\nRestored:", dst)

In [ ]:
from pathlib import Path
import xml.etree.ElementTree as ET

xml_dir = Path(
    "/content/Open_Duck_Playground/"
    "playground/open_duck_mini_v2/xmls"
)

src = xml_dir / "scene_delivery_tea_v2.xml"
dst = xml_dir / "scene_delivery_tea_slide.xml"

if not src.is_file():
    raise FileNotFoundError(src)

tree = ET.parse(src)
root = tree.getroot()

names = {
    "delivery_tray_surface",
    "yunomi_bottom_v2",
}

# 陶器の側壁も変更
count = 0

for geom in root.iter("geom"):
    name = geom.get("name", "")

    if (
        name in names
        or name.startswith("yunomi_wall_v2_")
    ):
        geom.set("friction", "0.4 0.005 0.0001")
        count += 1

if count != 18:
    raise RuntimeError(
        f"想定した18個のgeomではありません: {count}"
    )

tree.write(
    dst,
    encoding="unicode",
    xml_declaration=True
)

print("Created:", dst)
print("Updated geoms:", count)

In [ ]:
from pathlib import Path

root = Path("/content/Open_Duck_Playground")

src = root / "record_delivery_tea_v2.py"
dst = root / "record_delivery_tea_slide.py"

code = src.read_text()

old = "scene_delivery_tea_v2.xml"
new = "scene_delivery_tea_slide.xml"

if old not in code:
    raise RuntimeError("XML指定が見つかりません")

dst.write_text(code.replace(old, new))

print("Created:", dst)

In [ ]:
from pathlib import Path
import shutil

root = Path("/content/Open_Duck_Playground")

src = Path(
    "/content/drive/MyDrive/OpenDuckRL/"
    "delivery_140m/source/record_delivery_tea_v2.py"
)

dst = root / "record_delivery_tea_v2.py"

if not src.is_file():
    raise FileNotFoundError(src)

shutil.copy2(src, dst)

print("Restored:", dst)

In [ ]:
from pathlib import Path

root = Path("/content/Open_Duck_Playground")

src = root / "record_delivery_tea_v2.py"
dst = root / "record_delivery_tea_slide.py"

code = src.read_text()

old = "scene_delivery_tea_v2.xml"
new = "scene_delivery_tea_slide.xml"

if old not in code:
    raise RuntimeError("XML指定が見つかりません")

dst.write_text(code.replace(old, new))

print("Created:", dst)

In [ ]:
from pathlib import Path

root = Path("/content/Open_Duck_Playground")

for name in [
    "pyproject.toml",
    "uv.lock",
    ".venv",
    "record_delivery_tea_slide.py",
    "walk_140m.onnx",
]:
    p = root / name
    print(f"{name}: {'OK' if p.exists() else 'NG'}")

In [ ]:
%cd /content

!git clone https://github.com/apirrone/Open_Duck_Playground.git /content/Open_Duck_Playground_fresh

In [ ]:
%cd /content/Open_Duck_Playground

!uv add mujoco

In [ ]:
from pathlib import Path

fresh = Path("/content/Open_Duck_Playground_fresh")

for p in sorted(fresh.rglob("pyproject.toml")):
    print(p.relative_to(fresh))

In [ ]:
%cd /content/Open_Duck_Playground

!uv venv .venv
!uv pip install --python .venv/bin/python \
    mujoco \
    "imageio[ffmpeg]" \
    onnxruntime \
    numpy

In [ ]:
!./.venv/bin/python -c "import mujoco, imageio, onnxruntime, numpy; print('All imports OK')"

In [ ]:
%cd /content/Open_Duck_Playground

!MUJOCO_GL=egl ./.venv/bin/python record_delivery_tea_slide.py \
    --model /content/Open_Duck_Playground/walk_140m.onnx \
    --output /content/delivery_tea_slide.mp4 \
    --seconds 10

In [ ]:
from IPython.display import Video, display

display(Video(
    "/content/delivery_tea_slide.mp4",
    embed=True,
    width=800,
))

In [ ]:
from pathlib import Path

root = Path("/content/Open_Duck_Playground")
fresh = Path("/content/Open_Duck_Playground_fresh")

for name, path in {
    "作業ディレクトリ": root,
    "元リポジトリのコピー": fresh,
    "環境定義": root / "pyproject.toml",
    "仮想環境": root / ".venv/bin/python",
    "録画コード": root / "record_delivery_tea_slide.py",
    "歩行モデル": root / "walk_140m.onnx",
    "摩擦変更XML": root / "playground/open_duck_mini_v2/xmls/scene_delivery_tea_slide.xml",
}.items():
    print(f"{name}: {'OK' if path.exists() else 'NG'}")

In [ ]:
from pathlib import Path
import shutil

root = Path("/content/Open_Duck_Playground")

src = Path(
    "/content/drive/MyDrive/OpenDuckRL/"
    "soccer_140m/model/walk_140m.onnx"
)

dst = root / "walk_140m.onnx"

if not src.is_file():
    raise FileNotFoundError(src)

shutil.copy2(src, dst)

print("Model restored:", dst)
print("Size:", round(dst.stat().st_size / 1024**2, 2), "MB")

In [ ]:
%cd /content/Open_Duck_Playground

!./.venv/bin/python -c "import mujoco, imageio, onnxruntime; print('All imports OK')"

In [ ]:
%cd /content/Open_Duck_Playground

!MUJOCO_GL=egl ./.venv/bin/python record_delivery_tea_slide.py \
    --model /content/Open_Duck_Playground/walk_140m.onnx \
    --output /content/delivery_tea_slide.mp4 \
    --seconds 10

In [ ]:
from pathlib import Path

locations = [
    Path("/content/Open_Duck_Playground"),
    Path("/content/Open_Duck_Playground_fresh"),
]

for root in locations:
    print(f"\n=== {root} ===")
    for p in root.rglob("mujoco_infer.py"):
        print(p)

In [ ]:
from pathlib import Path
import shutil

root = Path("/content/Open_Duck_Playground")

dst = (
    root
    / "playground/open_duck_mini_v2/mujoco_infer.py"
)

candidates = list(
    Path("/content/Open_Duck_Playground_fresh").rglob(
        "mujoco_infer.py"
    )
)

if len(candidates) == 1:
    dst.parent.mkdir(parents=True, exist_ok=True)
    shutil.copy2(candidates[0], dst)
    print("Restored:", dst)
else:
    print("復元候補:", candidates)

In [ ]:
%cd /content/Open_Duck_Playground

!./.venv/bin/python -c "import playground.open_duck_mini_v2.mujoco_infer; print('mujoco_infer: OK')"

In [ ]:
from pathlib import Path
import shutil

src = Path("/content/Open_Duck_Playground_fresh")
dst = Path("/content/Open_Duck_Playground")

if not (src / "playground").is_dir():
    raise FileNotFoundError(src / "playground")

if not dst.is_dir():
    raise FileNotFoundError(dst)

copied = []

for file in (src / "playground").rglob("*"):
    if not file.is_file():
        continue

    # __pycache__などは除外
    if "__pycache__" in file.parts:
        continue

    relative = file.relative_to(src)
    target = dst / relative

    if not target.exists():
        target.parent.mkdir(parents=True, exist_ok=True)
        shutil.copy2(file, target)
        copied.append(str(relative))

print(f"Restored {len(copied)} files")

for name in copied[:30]:
    print(" +", name)

In [ ]:
%cd /content/Open_Duck_Playground

!./.venv/bin/python -c "import playground.open_duck_mini_v2.mujoco_infer; print('mujoco_infer: OK')"

In [ ]:
%cd /content/Open_Duck_Playground

!uv pip install --python .venv/bin/python "jax[cpu]"

In [ ]:
!./.venv/bin/python -c "import playground.open_duck_mini_v2.mujoco_infer; print('mujoco_infer: OK')"

In [ ]:
%cd /content/Open_Duck_Playground

!uv pip install --python .venv/bin/python ml-collections

In [ ]:
!./.venv/bin/python -c "import playground.open_duck_mini_v2.mujoco_infer; print('mujoco_infer: OK')"

In [ ]:
from pathlib import Path

fresh = Path("/content/Open_Duck_Playground_fresh")

for name in ["pyproject.toml", "uv.lock", "requirements.txt"]:
    p = fresh / name
    print(f"{name}: {'OK' if p.is_file() else 'NG'}")

In [ ]:
from pathlib import Path
import tomllib

src = Path("/content/Open_Duck_Playground_fresh/pyproject.toml")

with src.open("rb") as f:
    config = tomllib.load(f)

project = config.get("project", {})

print("Python:", project.get("requires-python"))
print("\nDependencies:")
for dep in project.get("dependencies", []):
    print(" -", dep)

print("\nOptional dependency groups:")
print(list(project.get("optional-dependencies", {}).keys()))

In [ ]:
from pathlib import Path
import shutil

src = Path("/content/Open_Duck_Playground_fresh/pyproject.toml")
dst = Path("/content/Open_Duck_Playground/pyproject.toml")

if not dst.exists():
    shutil.copy2(src, dst)
    print("Restored:", dst)
else:
    print("Already exists:", dst)

In [ ]:
%cd /content/Open_Duck_Playground

!uv pip install --python .venv/bin/python -e .

In [ ]:
%cd /content/Open_Duck_Playground

!uv pip install --python .venv/bin/python -e .

In [ ]:
!./.venv/bin/python -c "import playground.open_duck_mini_v2.mujoco_infer; print('mujoco_infer: OK')"

In [ ]:
!MUJOCO_GL=egl ./.venv/bin/python record_delivery_tea_slide.py \
    --model /content/Open_Duck_Playground/walk_140m.onnx \
    --output /content/delivery_tea_slide.mp4 \
    --seconds 10

In [ ]:
from IPython.display import Video, display

display(Video(
    "/content/delivery_tea_slide.mp4",
    embed=True,
    width=800
))

In [ ]:
from pathlib import Path
import shutil

root = Path("/content/Open_Duck_Playground")
backup = Path(
    "/content/drive/MyDrive/OpenDuckRL/delivery_140m"
)

files = {
    Path("/content/delivery_tea_slide.mp4"):
        backup / "videos/delivery_tea_slide.mp4",
    root / "record_delivery_tea_slide.py":
        backup / "source/record_delivery_tea_slide.py",
    root / "playground/open_duck_mini_v2/xmls/scene_delivery_tea_slide.xml":
        backup / "source/scene_delivery_tea_slide.xml",
}

for src, dst in files.items():
    if not src.is_file():
        raise FileNotFoundError(src)
    dst.parent.mkdir(parents=True, exist_ok=True)
    shutil.copy2(src, dst)
    print("Saved:", dst)

In [ ]:
from pathlib import Path
import xml.etree.ElementTree as ET

root_dir = Path("/content/Open_Duck_Playground")
xml_dir = root_dir / "playground/open_duck_mini_v2/xmls"

src = xml_dir / "scene_delivery_tea_slide.xml"
dst = xml_dir / "scene_delivery_tea_slide015.xml"

tree = ET.parse(src)
root = tree.getroot()

count = 0

for geom in root.iter("geom"):
    name = geom.get("name", "")

    if (
        name in {"delivery_tray_surface", "yunomi_bottom_v2"}
        or name.startswith("yunomi_wall_v2_")
    ):
        geom.set("friction", "0.1 0.005 0.0001")
        count += 1

assert count == 18, f"変更対象が想定と異なります: {count}"

tree.write(dst, encoding="unicode", xml_declaration=True)

print("Created:", dst)
print("Friction:", 0.10)
print("Updated geoms:", count)

In [ ]:
from pathlib import Path

root = Path("/content/Open_Duck_Playground")

src = root / "record_delivery_tea_slide.py"
dst = root / "record_delivery_tea_slide015.py"

code = src.read_text(encoding="utf-8")

old = "scene_delivery_tea_slide.xml"
new = "scene_delivery_tea_slide015.xml"

if old not in code:
    raise RuntimeError("XML指定が見つかりません")

dst.write_text(code.replace(old, new), encoding="utf-8")

print("Created:", dst)

In [ ]:
%cd /content/Open_Duck_Playground

!MUJOCO_GL=egl ./.venv/bin/python record_delivery_tea_slide015.py \
    --model /content/Open_Duck_Playground/walk_140m.onnx \
    --output /content/delivery_tea_slide015.mp4 \
    --seconds 10

In [ ]:
from IPython.display import Video, display

display(Video(
    "/content/delivery_tea_slide015.mp4",
    embed=True,
    width=800
))

In [ ]:
from pathlib import Path
import shutil

root = Path("/content/Open_Duck_Playground")

backup = Path(
    "/content/drive/MyDrive/OpenDuckRL/delivery_140m/final"
)

# 保存対象
files = {
    "video": [
        Path("/content/delivery_tea_slide01.mp4"),
    ],
    "source": [
        root / "record_delivery_tea_slide01.py",
        root / "playground/open_duck_mini_v2/xmls/scene_delivery_tea_slide01.xml",
    ],
    "model": [
        root / "walk_140m.onnx",
    ],
}

# 保存先フォルダを作成
for folder in files:
    (backup / folder).mkdir(parents=True, exist_ok=True)

# ファイルをコピー
for folder, paths in files.items():
    for src in paths:
        if not src.is_file():
            print("NOT FOUND:", src)
            continue

        dst = backup / folder / src.name
        shutil.copy2(src, dst)

        print("Saved:", dst)

print("\nBackup complete:", backup)

In [ ]:
%%writefile /content/Open_Duck_Playground/create_delivery_head.py
from pathlib import Path
import xml.etree.ElementTree as ET
import numpy as np
import mujoco

ROOT = Path("/content/Open_Duck_Playground")
XML_DIR = ROOT / "playground/open_duck_mini_v2/xmls"
BACKUP = Path("/content/drive/MyDrive/OpenDuckRL/delivery_140m/source")

OUTPUT = XML_DIR / "scene_delivery_tea_head.xml"

FRICTION = 0.1
HEAD_GAP = 0.002
CUP_GAP = 0.002

# 既存の摩擦0.1モデルを優先
candidates = [
    XML_DIR / "scene_delivery_tea_slide015.xml",
    XML_DIR / "scene_delivery_tea_slide01.xml",
    XML_DIR / "scene_delivery_tea_slide.xml",
    BACKUP / "scene_delivery_tea_v2.xml",
]

source = next((p for p in candidates if p.is_file()), None)

if source is None:
    raise FileNotFoundError(
        "配膳XMLがありません。Driveのバックアップを確認してください。"
    )

print("Source:", source)

tree = ET.parse(source)
xml_root = tree.getroot()

head = xml_root.find(".//body[@name='head_assembly']")
tray = xml_root.find(".//body[@name='delivery_tray']")
cup = xml_root.find(".//body[@name='yunomi']")

if head is None or tray is None or cup is None:
    raise RuntimeError("head / tray / cup が見つかりません")

# 変更前の姿勢を取得
original = mujoco.MjModel.from_xml_path(str(source))
data = mujoco.MjData(original)

if original.nkey:
    mujoco.mj_resetDataKeyframe(original, data, 0)

mujoco.mj_forward(original, data)

head_id = original.body("head_assembly").id
head_pos = data.xpos[head_id].copy()
head_rot = data.xmat[head_id].reshape(3, 3).copy()

# 頭部に直接属するgeomから頭頂部の高さを推定
geom

In [ ]:
%cd /content/Open_Duck_Playground

from pathlib import Path
import shutil
import subprocess

root = Path("/content/Open_Duck_Playground")
backup = Path("/content/drive/MyDrive/OpenDuckRL")
xml_dir = root / "playground/open_duck_mini_v2/xmls"

# 既存のONNXを復元
onnx = root / "walk_140m.onnx"

if not onnx.is_file():
    src = backup / "soccer_140m/model/walk_140m.onnx"
    if not src.is_file():
        raise FileNotFoundError(src)
    shutil.copy2(src, onnx)

# 既存の録画スクリプトを取得
script_candidates = [
    root / "record_delivery_tea_slide015.py",
    root / "record_delivery_tea_slide.py",
    root / "record_delivery_tea_v2.py",
    backup / "delivery_140m/source/record_delivery_tea_v2.py",
]

record_src = next(
    (p for p in script_candidates if p.is_file()),
    None
)

if record_src is None:
    raise FileNotFoundError("録画スクリプトがありません")

# 頭部取り付けXMLを生成
subprocess.run(
    [str(root / ".venv/bin/python"), "create_delivery_head.py"],
    cwd=root,
    check=True,
)

# 録画スクリプトを新しいXML参照に変更
code = record_src.read_text(encoding="utf-8")

old_names = [
    "scene_delivery_tea_slide015.xml",
    "scene_delivery_tea_slide01.xml",
    "scene_delivery_tea_slide.xml",
    "scene_delivery_tea_v2.xml",
]

found = [name for name in old_names if name in code]

if len(found) != 1:
    raise RuntimeError(
        f"録画スクリプト内のXML参照が想定外です: {found}"
    )

code = code.replace(
    found[0],
    "scene_delivery_tea_head.xml",
)

record_dst = root / "record_delivery_tea_head.py"
record_dst.write_text(code, encoding="utf-8")

print("\nRecord script:", record_dst)
print("Model:", onnx)
print("Ready")

In [ ]:
%cd /content/Open_Duck_Playground

!MUJOCO_GL=egl ./.venv/bin/python record_delivery_tea_head.py \
    --model /content/Open_Duck_Playground/walk_140m.onnx \
    --output /content/delivery_tea_head.mp4 \
    --seconds 10

from IPython.display import Video, display

video = "/content/delivery_tea_head.mp4"

if Path(video).is_file():
    display(Video(video, embed=True, width=800))

In [ ]:
from pathlib import Path
import shutil

root = Path("/content/Open_Duck_Playground")
dst_root = Path(
    "/content/drive/MyDrive/OpenDuckRL/"
    "delivery_140m/head_final"
)

files = {
    "video/delivery_tea_head.mp4":
        Path("/content/delivery_tea_head.mp4"),

    "source/create_delivery_head.py":
        root / "create_delivery_head.py",

    "source/record_delivery_tea_head.py":
        root / "record_delivery_tea_head.py",

    "source/scene_delivery_tea_head.xml":
        root / "playground/open_duck_mini_v2/xmls/scene_delivery_tea_head.xml",

    "model/walk_140m.onnx":
        root / "walk_140m.onnx",
}

# 一部だけ保存する状態を避ける
missing = [str(p) for p in files.values() if not p.is_file()]
if missing:
    raise FileNotFoundError(
        "保存対象が不足しています:\n" + "\n".join(missing)
    )

for name, src in files.items():
    dst = dst_root / name
    dst.parent.mkdir(parents=True, exist_ok=True)
    shutil.copy2(src, dst)
    print("Saved:", dst)

print("\nBackup complete:", dst_root)